In [ ]:
/**
 * REAL-TIME UPDATES IN SYSTEM DESIGN:
 * - Imagine you're texting a friend. 
 * - If they get your message instantly, that feels normal. 
 * - If they got it 10 minutes late, you'd be confused.
 * - That "feels instant" feeling is what "real-time updates" is all about.
 * - But here's the first big lesson: "real-time" doesn't mean ONE thing.
 *   - A collaborative doc (like Google Docs) needs updates in ~100ms
 *   - A chat app needs updates in ~1 second
 *   - A ride-sharing app tracking a driver needs updates every ~5 seconds
 *   - A dashboard showing stats can wait ~30 seconds
 * - So "make it real-time" is a vague request. 
 * - The REAL skill is figuring out HOW FAST it actually needs to be, then picking 
 *   the cheapest tool that hits that speed. 
 * - Overbuilding (using the fanciest tech when you don't need it) wastes time and money.
 * - That's the whole theme of this file.
*/

/**
 * WHY NORMAL WEB REQUESTS AREN'T ENOUGH
 * - Normally, the web works like this:
 *   1. You (the client) ask the server for something.
 *   2. The server answers.
 *   3. Done. Conversation over.
 * - This is called "request/response." 
 * - It's like knocking on someone's door, asking a question, getting an answer, 
 *   and walking away.
 *
 * Problem: 
 * - What if something changes AFTER you walk away? 
 * - You won't know unless you knock again. 
 * - That's the core problem real-time systems need to solve — how does the client 
 *   find out about NEW stuff without having to keep guessing and re-asking?
 *
 * Solution:
 * - There are 4 classic answers to this problem. 
 * - Think of them as a ladder, going from "simple but wasteful" to "powerful but 
 *   complicated":
 *   1. Short Polling      -> keep asking "anything new?" over and over
 *   2. Long Polling       -> ask once, but the server waits to answer
 *   3. Server-Sent Events -> open a one-way pipe, server pushes to you
 *   4. WebSockets         -> open a two-way pipe, both sides talk freely
*/

/**
 * APPROACH 1: SHORT POLLING — "Are we there yet? Are we there yet?"
 * - This is the most obvious idea a beginner would come up with: 
 *   just keep asking the server, over and over, on a timer.
 *
 *   Client: "Any updates?"        Server: "Nope."
 *   (wait 5 seconds)
 *   Client: "Any updates?"        Server: "Nope."
 *   (wait 5 seconds)
 *   Client: "Any updates?"        Server: "Yes! Here you go."
 *
 * ARCHITECTURE PICTURE:
 *
 *      [Client] --- "got anything new?" ---> [Server]
 *      [Client] <--------- "nope" ----------- [Server]
 *              (wait 5 sec, repeat forever)
 *
 * HOW IT WORKS (step by step):
 *   - The client sets a timer, like "every 5 seconds."
 *   - Every time the timer fires, it sends a normal HTTP request.
 *   - The server checks: "do I have anything new for this client?"
 *   - It replies either with the new data, or with "nothing new."
 *   - The client waits again and repeats.
 *
 * THE BIG TRADE-OFF (this is the important part):
 *   - Poll OFTEN (every 1 second)  -> updates feel fast, but you're sending
 *     tons of requests, most of which come back empty. Wastes server power
 *     and drains phone batteries.
 *   - Poll RARELY (every 30 seconds) -> saves resources, but updates feel
 *     slow and laggy.
 *   You can't have both fast AND cheap with this method. That's the deal.
 *
 *   Real numbers: if 1 million people are polling every 5 seconds, that's
 *   about 200,000 requests EVERY SECOND hitting your servers — and most
 *   of them just say "nothing new." That's a lot of wasted effort.
 *
 * WHEN TO USE IT:
 *   - Good for: checking email, refreshing a dashboard, background sync —
 *     places where a few seconds (or a minute) of delay is totally fine.
 *   - Bad for: chat apps, live location tracking, trading apps, games —
 *     anywhere speed actually matters.
 *
 * PROS:
 *   + Super simple to build (just a timer + a normal request)
 *   + Works literally everywhere, no special browser features needed
 *   + Server doesn't have to "remember" anything between requests
 *   + Very easy to debug — every request stands alone
 *
 * CONS:
 *   - Wastes bandwidth (lots of "nothing new" responses)
 *   - Updates are delayed on average by half your polling interval
 *   - Doesn't scale well with lots of users
 *   - Bad for phone battery life (constant network activity)
*/

/**
 * APPROACH 2: LONG POLLING — "I'll wait right here until you have news"
 * - Long polling is a clever twist: instead of the server saying "nope,
 *   nothing new, ask me again later," the server just... doesn't answer yet.
 * - It holds the door open and waits until it actually has something to say.
 *
 *   Client: "Any updates?"
 *   Server: (doesn't respond yet... waiting... waiting...)
 *   Server: (something happens!) "Yes! Here's the update!"
 *   Client: immediately asks again: "Any updates?"
 *   Server: (waits again...)
 *
 * ARCHITECTURE PICTURE:
 *
 *   [Client] ---"any updates?"---> [Server] (holds request open, no reply yet)
 *                                     |
 *                          (new data shows up somewhere)
 *                                     |
 *   [Client] <----"here's data!"----- [Server]  (finally responds)
 *   [Client] ---"any updates?"------> [Server]  (client immediately re-asks)
 *
 * WHY THIS IS BETTER:
 * - Instead of asking "anything new?" every 5 seconds regardless of
 *   whether anything happened, the client asks ONCE and just waits.
 * - The server only replies when it truly has something to say 
 *   (or after a timeout, like 30-60 seconds, just so the connection
 *   doesn't hang forever). This means way fewer wasted "nope" responses.
 *
 * THE HARD PART (behind the scenes):
 * - Here's a tricky bit for the SERVER: 
 *   if it has 10,000 clients all waiting for updates, it CANNOT have 10,000 threads
 *   just sitting there doing nothing but waiting — that wastes memory and CPU management.
 * - Instead, real systems use something called "pub/sub" (publish and subscribe) — 
 *   think of it like a bulletin board:
 *
 *   ARCHITECTURE PICTURE (behind the scenes):
 *
 *        [Something happens] --publishes to--> [Topic: "updates"]
 *                                                      |
 *                          notifies whoever is "subscribed" / waiting
 *                                  /            |            \
 *                        [Handler 1]      [Handler 2]     [Handler 3]
 *                        (Client A         (Client B       (Client C
 *                         waiting)          waiting)        waiting)
 *
 *   Each waiting client "registers interest" and then goes idle (without
 *   blocking a whole thread). When new data is published to the topic,
 *   only the relevant waiting clients get woken up and answered.
 *
 *   Important nuance: this pub/sub board is usually TEMPORARY (ephemeral).
 *   If a client disconnects and reconnects later, and needs to catch up
 *   on what it missed, you need something more permanent — like a
 *   database or a log of past events — not just the bulletin board.
 *
 * WHEN TO USE IT:
 *   - Good for: notifications, activity feeds, medium-scale chat, progress
 *     updates — things that need "pretty quick" updates but don't need
 *     the client to send messages back frequently.
 *   - Consider something else if: messages are very frequent, communication
 *     needs to go both ways a lot, or you have a massive number of users.
 *
 * PROS:
 *   + Feels much faster than short polling, way fewer wasted requests
 *   + Still just plain HTTP — works through most firewalls/proxies fine
 *   + Simpler to build than the fancier options below
 *
 * CONS:
 *   - Each waiting client ties up a server connection (costs resources)
 *   - Needs an "async" server design to handle many waiting clients at once
 *   - After every answer, the client has to open a brand new request,
 *     which adds a little extra delay each cycle
 *   - Ordering can get messy if requests and responses overlap awkwardly
 *   - Can't use CDNs to help here — the responses are dynamic, not cacheable
*/

/**
 * APPROACH 3: SERVER-SENT EVENTS (SSE) — "I'll just tell you when stuff happens"
 * - With SSE, the client doesn't ask over and over anymore. 
 * - It opens ONE connection and just leaves it open. 
 * - Whenever the server has news, it pushes it straight down that same open pipe — 
 *   no new request needed.
 *
 * - Think of it like leaving a phone call open with a friend who says 
 *   "hey, new thing happened" whenever it happens, instead of you calling them
 *   again and again to check in.
 *
 * ARCHITECTURE PICTURE:
 *
 *   [Client] ---"open a stream please"---> [Server]
 *   [Client] <----------- event: price-update -------- [Server]
 *   [Client] <----------- event: trade --------------- [Server]
 *   [Client] <----------- event: price-update -------- [Server]
 *                  (all over ONE connection that stays open)
 *
 * HOW IT WORKS:
 *   - The browser has a built-in tool for this called "EventSource" —
 *     you basically just tell it a URL, and it handles opening the
 *     connection, listening, and even reconnecting if it drops.
 *   - The server responds with a special "I'm going to keep streaming"
 *     content type, and then it can push little text messages any time.
 *
 * WHAT AN SSE MESSAGE LOOKS LIKE:
 *   It's just simple text, like this:
 *
 *     event: price-update
 *     id: 12345
 *     data: { symbol: "AAPL", price: 150.25 }
 *
 *   - "event" -> what TYPE of update this is (so the client can route it)
 *   - "id"    -> a unique ID, used for "resuming" after a disconnect
 *   - "data"  -> the actual info being sent (usually JSON)
 *
 *   Because it's just plain text, you can literally watch it happen in
 *   your browser's network tab. Very easy to debug.
 *
 * THE COOL PART — AUTO-RECONNECT:
 *   If the connection drops (network hiccup, server restart, etc), the
 *   BROWSER automatically tries to reconnect for you. And it even tells
 *   the server "hey, I last saw id 102" using a header called
 *   "Last-Event-ID", so the server can (if it kept a log) resume exactly
 *   where it left off instead of the client missing messages.
 *
 *   ARCHITECTURE PICTURE (reconnect flow):
 *
 *      [Client] <--- events 100, 101, 102 --- [Server]
 *      *connection drops*
 *      [Client] ---"reconnect, Last-Event-ID: 102"---> [Server]
 *      [Client] <--- events 103, 104 --- [Server] (picks up where it left off)
 *
 *   Note: this auto-reconnect is helpful, but it's not a magic guarantee.
 *   The SERVER still has to actually store recent events somewhere so it
 *   CAN resume — the browser feature alone doesn't create durability.
 *
 * THE BIG LIMITATION:
 *   SSE is ONE-WAY ONLY. The server can push to the client, but the
 *   client can't send messages back through the same pipe. If the client
 *   needs to talk back, it just makes a normal, separate HTTP request
 *   (like a regular POST) alongside the SSE stream.
 *
 * WHEN TO USE IT:
 *   - Great for: stock tickers, live sports scores, notification streams,
 *     build/deploy progress, social feeds — basically anything where the
 *     server is mostly "telling" the client stuff, not the other way around.
 *   - Not great for: chat, multiplayer games, collaborative editing —
 *     these need frequent two-way conversation, which SSE isn't built for.
 *
 * PROS:
 *   + Browser handles it for you (EventSource API) — very little code needed
 *   + Automatic reconnect + resume support built in
 *   + Plain text = easy to debug
 *   + Lighter weight than WebSockets (simpler protocol, less server cost)
 *
 * CONS:
 *   - One-way only (server -> client)
 *   - Browsers historically limit ~6 connections per site on old HTTP/1.1
 *     (HTTP/2 mostly fixes this)
 *   - Can't set custom headers (like an Authorization token) on the
 *     built-in browser API — usually need cookies or a token in the URL
 *   - Text only — sending binary data (like images) needs extra encoding
*/

/**
 * APPROACH 4: WEBSOCKETS — "A phone line where both people can talk anytime"
 * - WebSockets are the "full power" option. 
 * - Once the connection opens, BOTH the client and the server can send messages
 *   to each other, at any time, without waiting for a request first. 
 * - No more "who goes first" — it's a real two-way conversation.
 *
 * ARCHITECTURE PICTURE:
 *
 *   [Client] <==== persistent two-way connection ====> [Server]
 *
 *   [Client] ---> "user started typing"
 *   [Server] ---> "here's a new chat message"
 *   [Client] ---> "user sent a message"
 *   [Server] ---> "message delivered!"
 *          (either side can send, anytime, no fixed order)
 *
 * HOW THE CONNECTION STARTS:
 * - It actually begins as a normal HTTP request! 
 * - The client says "hey, can we switch protocols to WebSocket?" and 
 *   if the server agrees, it replies "101 Switching Protocols." 
 * - From that point on, it's no longer HTTP — it's a WebSocket connection, and 
 *   both sides can just send data freely, called "frames," with very little overhead
 *   per message (compared to sending a full HTTP request every time).
 *
 * WHY THIS IS DIFFERENT FROM EVERYTHING ABOVE:
 *   +---------------------+---------------------------+----------------------+
 *   | Thing               | HTTP-based options        | WebSocket            |
 *   |---------------------|---------------------------|----------------------|
 *   | Who starts talking  | Client always goes first  | Either side, anytime |
 *   | Pattern             | Ask -> Answer             | Free-flowing chat    |
 *   | Per-message cost    | Full headers every time   | Very lightweight     |
 *   +---------------------+---------------------------+----------------------+
 *
 * THE BIG CHALLENGE: SCALING
 * - This is where things get harder. 
 * - Each connected client keeps ONE connection open to ONE specific server, 
 *   and that connection uses up real resources (memory, and something called 
 *   a "file descriptor" — basically a slot the operating system uses to track 
 *   an open connection). 
 * - Servers can usually handle tens of thousands of these comfortably, and 
 *   highly-tuned setups can handle up to ~1 million — but you have to actually
 *   plan and configure for it.
 *
 *   Here's the tricky problem this creates:
 *   - Imagine Alice is connected to Server 1, and Bob is connected to Server 2. 
 *   - Alice wants to send Bob a message. 
 *   - Server 1 has NO idea how to reach Bob directly — Bob isn't even connected to it!
 *
 *   ARCHITECTURE PICTURE (the cross-server messaging problem):
 *
 *        [Alice] --- connected to ---> [Server 1]
 *        [Bob]   --- connected to ---> [Server 2]
 *
 *        Alice sends a message meant for Bob... now what?
 *
 *   THE FIX: a shared "message bus" (like Redis or Kafka) sits in the
 *   middle. Server 1 publishes Alice's message to the bus, and the bus
 *   makes sure it reaches Server 2, which then delivers it to Bob.
 *
 *   ARCHITECTURE PICTURE (with the message bus):
 *
 *      [Alice]--[Server 1]--publish-->[Message Bus]--deliver-->[Server 2]--[Bob]
 *
 *   There are two ways to route messages through the bus:
 *     - BROADCAST: every server hears every message and just ignores the
 *       ones that aren't for its own users. Simple to build, but wasteful.
 *     - TARGETED: look up exactly which server holds Bob's connection,
 *       and only send it there. More efficient, but needs a lookup table
 *       ("connection registry") that tracks WHO is connected to WHERE:
 *
 *         user:alice -> server1
 *         user:bob   -> server2
 *         user:carol -> server1
 *
 *       You also need to keep this registry clean — using things like
 *       heartbeats (regular "are you still there?" pings) and expiration
 *       timers, so it doesn't point to a server that already crashed.
 *
 * WHEN TO USE IT:
 *   - Great for: chat apps, multiplayer games, collaborative editing
 *     (like Figma/Google Docs), live trading — anywhere BOTH sides send
 *     frequent messages.
 *   - Overkill for: news feeds, notifications, live scores — those are
 *     mostly one-directional, so SSE is usually simpler and cheaper.
 *
 * PROS:
 *   + True two-way communication, either side can talk anytime
 *   + Low overhead per message (good for high-frequency chatting)
 *   + Can send efficient binary data, not just text
 *   + Great fit for interactive, fast-paced apps
 *
 * CONS:
 *   - Much more complex to build and run than the HTTP-based options
 *   - Needs extra infrastructure (like a message bus) to scale across
 *     multiple servers
 *   - Keeping track of "who's connected where" (stateful) is harder than
 *     stateless HTTP
 *   - Some corporate firewalls block WebSocket traffic
 *   - The browser does NOT auto-reconnect for you like SSE does — you
 *     have to build that yourself (or use a helper library)
 *   - Harder to debug since you can't just look at simple request/response
 *     pairs
*/

/**
 * SIDE-BY-SIDE COMPARISON:
 * 1. Who talks first
 *    - Short Polling: Client repeatedly
 *    - Long Polling : Client (then waits)
 *    - SSE          : Client opens, server sends
 *    - WebSocket    : Either side
 * 2. Speed/Latency 
 *    - Short Polling: Depends on poll interval
 *    - Long Polling : Fast while waiting
 *    - SSE          : Fast
 *    - WebSocket    : Either side
 * 3. Direction
 *    - Short Polling: One-way ask
 *    - Long Polling : One-way ask
 *    - SSE          : Server -> Client only
 *    - WebSocket    : Both Ways
 * 4. Server Complexity
 *    - Short Polling: Simplest
 *    - Long Polling : Medium
 *    - SSE          : Medium
 *    - WebSocket    : Hardest
 * 5. Scaling Difficulty
 *    - Short Polling: Easy
 *    - Long Polling : Medium
 *    - SSE          : Medium
 *    - WebSocket    : Hard
 * 6. Auto-reconnect
 *    - Short Polling: Just poll again
 *    - Long Polling : Manual loop
 *    - SSE          : Browser does it for you
 *    - WebSocket    : You build it yourself
 * 7. Works through firewalls?
 *    - Short Polling: Yes
 *    - Long Polling : Yes
 *    - SSE          : Yes
 *    - WebSocket    : Blocked
*/

/**
 * HOW TO CHOOSE (the decision-making checklist):
 * - Before picking any of the four, ask yourself these questions — in order:
 *   1. How fast does this REALLY need to be?
 *      - If a few seconds delay is fine and you don't have a ton of users,
 *        short polling might be all you need. 
 *      - Don't overbuild!
 *
 *   2. Do I have a LOT of users?
 *      - Even if delay is okay, tons of users polling constantly gets expensive fast.
 *      - This nudges you toward long polling or SSE.
 *
 *   3. Does the client need to send frequent messages back?
 *      - If YES (like chat or games)     -> WebSockets are probably worth it.
 *      - If NO (mostly server -> client) -> SSE is usually simpler.
 *
 *   4. Do I need replay, ordering, or guaranteed delivery?
 *      - None of these 4 approaches solve that by themselves! 
 *      - You still need extra tools: 
 *        - sequence numbers, 
 *        - a durable log/database, and 
 *        - de-duplication logic on the client.
 *
 * COMMON BEGINNER MISTAKE:
 * - Jumping straight to WebSockets because they sound the most powerful,
 *   then later realizing the REAL problem was about message replay, ordering, or
 *   handling lots of "fan-out" — problems that WebSockets alone don't solve anyway.
 * - Start simple. 
 * - Add complexity only when you have a real reason to.
*/

/**
 * THE "MAKE IT ACTUALLY RELIABLE" CHECKLIST
 * - No matter which of the 4 approaches you pick, real systems ALWAYS need
 *   to handle these problems. 
 * - Skipping them is what turns a nice demo into a broken product.
 *
 * 1. RECONNECTING WHEN THINGS DROP (exponential backoff)
 *    - Connections WILL drop — phones switch WiFi, servers restart, etc.
 *    - Don't just retry immediately over and over (that can overload the
 *      server right when it's already struggling). 
 *    - Instead, wait a little longer each time you fail, like this:
 *
 *      Try 1: wait 1 second
 *      Try 2: wait 2 seconds
 *      Try 3: wait 4 seconds
 *      Try 4: wait 8 seconds
 *      Try 5: wait 16 seconds
 *      Try 6+: cap it at 30 seconds
 *
 *    - Also add a little bit of randomness ("jitter") so that if 10,000
 *      clients all disconnected at once, they don't all retry at the EXACT
 *      same moment and overwhelm the server again.
 *
 * 2. HEARTBEATS (checking if the other side is still alive)
 *    - A silent connection and a DEAD connection look exactly the same on
 *      the network — you can't tell the difference just by waiting. 
 *    - So you send a tiny "ping" message every so often (like every 30 seconds)
 *      and expect a "pong" back. 
 *    - No pong? Assume it's dead, reconnect.
 *
 * 3. MESSAGE ORDERING & DUPLICATES
 *    Messages can arrive out of order, or even arrive twice (especially
 *    after a reconnect). Fix this by giving every message a sequence
 *    number. The client tracks "the last number I've seen," buffers
 *    anything that arrives out of order, and ignores duplicates.
 *
 * 4. SLOW CLIENTS (backpressure)
 *    Sometimes the server is producing updates faster than a client can
 *    process them (think: old phone on a slow connection). Options:
 *      - Drop unimportant messages (e.g., only keep the LATEST stock price)
 *      - Batch several updates into one bigger message
 *      - Just disconnect clients that fall too far behind
 *    Pick based on whether losing a message is okay (stock prices: often
 *    fine) or not okay (chat messages: usually NOT fine).
 *
 * 5. SECURITY ON LONG-LIVED CONNECTIONS
 *    Normal HTTP checks your login token on every single request. But a
 *    WebSocket or SSE connection might stay open for HOURS — so what
 *    happens if your token expires while it's still connected?
 *      - Always use encrypted connections (wss:// and https://, never
 *        the unencrypted versions)
 *      - Watch out for a sneaky attack where a bad website secretly opens
 *        a WebSocket to your server using the victim's saved login
 *        cookies — defend by checking the "Origin" header
 *      - Since SSE can't send custom auth headers, be careful with
 *        putting tokens in the URL — they can leak into server logs
 *
 * 6. LOAD BALANCERS AND DEPLOYS
 *    Normal HTTP requests can go to ANY server, since each one is
 *    independent. But once a WebSocket connection is open, it's STUCK on
 *    that one server. This means:
 *      - Your load balancer needs to understand and properly pass through
 *        the WebSocket "upgrade" request (some older ones break this)
 *      - When you deploy new code and shut down a server, ALL its
 *        connections drop at once — so you want to gradually drain
 *        connections and roll out changes in small batches, not all at
 *        once, to avoid a huge reconnect stampede.
 *
 *
 * ============================================================================
 * REAL-WORLD EXAMPLES (so it's not just theory)
 * ============================================================================
 *
 *   App                    | What it uses          | Why
 *   -----------------------|-----------------------|---------------------------
 *   Slack                  | WebSocket             | Messages, presence, and
 *                          |                       | typing all flow both ways,
 *                          |                       | constantly
 *   Discord                | WebSocket + sequence  | Uses heartbeats + a
 *                          | numbers (resume)      | "resume" system to
 *                          |                       | recover missed events
 *   WhatsApp               | Persistent socket     | Needs instant push, so
 *                          |                       | keeps a long-lived
 *                          |                       | connection open
 *   Figma                  | WebSocket             | Multiple people editing
 *                          |                       | the same design need
 *                          |                       | constant two-way sync
 *   Coinbase / Binance     | WebSocket (feeds)     | Streams live prices —
 *                          |                       | mostly one-way, but uses
 *                          |                       | WebSocket for speed
 *   ChatGPT-style chat UIs | SSE                   | Streams the AI's answer
 *                          |                       | word-by-word — one-way,
 *                          |                       | so SSE is simpler and
 *                          |                       | cheaper than WebSocket
 *
 *   Fun fact: a LOT of engineers assume "AI streaming word-by-word" must
 *   use WebSockets because it feels so "live" — but it's usually SSE,
 *   because the data only flows ONE direction (server -> client).
 *
 *
 * ============================================================================
 * BONUS: TWO RELATED TOOLS WORTH KNOWING
 * ============================================================================
 *
 * 1) PUSH NOTIFICATIONS (Apple's APNs / Google's FCM)
 *    None of the 4 approaches above work if your app is closed or in the
 *    background — the phone's operating system literally kills the
 *    connection to save battery. That's what push notifications are for:
 *    the OS itself keeps a connection to Apple/Google, and your server
 *    just asks THEM to wake up the app and show an alert.
 *
 *    Real apps combine both: a live connection (WebSocket/SSE) while the
 *    app is open, and push notifications when it's closed.
 *
 * 2) WEBRTC
 *    For the absolute lowest latency, like video calls or screen sharing,
 *    apps use WebRTC. It connects two browsers DIRECTLY to each other
 *    (skipping the server when possible) using UDP, which is faster but
 *    doesn't guarantee every packet arrives. It's more complex to set up
 *    and mostly used specifically for audio/video, not general data.
 *
 *
 * ============================================================================
 * THE BIG TAKEAWAYS (if you remember nothing else, remember this)
 * ============================================================================
 *
 *   1. "Real-time" isn't one thing — figure out the ACTUAL speed you need
 *      before picking a tool.
 *
 *   2. Start simple. If short polling solves your problem, USE IT. Don't
 *      reach for WebSockets just because they sound impressive.
 *
 *   3. SSE is underrated — if data mostly flows server -> client, it's
 *      simpler than WebSockets and the browser handles reconnecting for you.
 *
 *   4. WebSockets are powerful, but they come with real operational cost
 *      (message buses, connection registries, scaling headaches). Only
 *      reach for them when you truly need fast, two-way communication.
 *
 *   5. The transport (which of the 4 you pick) is NOT the same thing as
 *      reliable delivery. You still need sequence numbers, durable
 *      storage/logs, deduplication, and backpressure handling regardless
 *      of which transport you choose.
 *
 *   6. Everything WILL eventually break — connections drop, servers
 *      restart, phones lose signal. Plan for reconnects (with backoff),
 *      heartbeats, and message ordering from day one, not as an
 *      afterthought.
 *
 *   The simplest solution that meets your actual requirements is usually
 *   the right one. Add complexity only once you have real evidence you
 *   need it.
 *
 * ============================================================================
 */

In [ ]:
/**
 * ====================================================================
 * ROUND 1: PROBLEM SAMAJHNA - "Real-time" ka Matlab Kya Hai?
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Design karo ek system jisme users ko FRESH information dikhe
 *    bina page refresh kiye - jaise chat message, driver location,
 *    ya notification. Kaise approach karoge?"
 * 
 * CANDIDATE:
 *   "Sir, sabse pehli galti jo log karte hain, woh yeh hai ki
 *    'REAL-TIME' ko EK SINGLE REQUIREMENT samajh lete hain. Lekin
 *    ASAL mein 'real-time' ka matlab CONTEXT pe depend karta hai:
 * 
 *      - Collaborative editor (jaise Google Docs) ke liye: ~100ms
 *      - Chat ke liye: ~1 second
 *      - Driver tracking (Uber) ke liye: ~5 seconds
 *      - Dashboard metrics ke liye: ~30 seconds
 * 
 *    Yeh TARGET LATENCY hi decide karti hai ki HUM KAUNSA TRANSPORT,
 *    KITNA COST, aur KAUNSA CORRECTNESS MODEL use karenge. Isliye
 *    main pehle REQUIREMENTS clarify karna chahunga, seedha
 *    'WebSockets use karunga' bolke shuru nahi karunga."
 * 
 * INTERVIEWER:
 *   "Traditional request/response model kyun kaafi nahi hai?"
 * 
 * CANDIDATE:
 *   "Kyunki us model mein CLIENT REQUEST karta hai, SERVER RESPOND
 *    karta hai, aur EXCHANGE KHATAM ho jata hai. Agar SERVER pe
 *    BAAD mein kuch CHANGE hota hai, CLIENT ko pata karne ke liye
 *    YA toh EK AUR REQUEST bhejni padegi, YA EK OPEN CHANNEL chahiye
 *    hoga jisse WOH pata chale. Isi problem ko solve karne ke liye
 *    Short Polling, Long Polling, SSE, aur WebSockets jaise
 *    approaches bane hain."
*/

/**
 * ====================================================================
 * ROUND 2: CLARIFYING QUESTIONS - Sahi Approach Chunne Se Pehle
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Theek hai, pucho jo pochna hai."
 * 
 * CANDIDATE (Clarifying Questions):
 *   1. "Kitni LATENCY chahiye - milliseconds, seconds, ya bas
 *       'eventually visible' chalega?"
 *   2. "Communication ONE-WAY hai (server-to-client) ya BIDIRECTIONAL hai 
 *       (dono taraf se)?"
 *   3. "Kitne CLIENTS honge aur PER SECOND kitne MESSAGES flow karenge?"
 *   4. "Messages DURABLE hone chahiye (REPLAY zaroori hai) ya
 *       SIRF live clients ko wake karna kaafi hai?"
 *   5. "ORDERING zaroori hai kya? (jaise chat messages, document edits, trades)"
 *   6. "RECONNECT hone par kya hoga - resume kaise karenge?"
 *   7. "Kya UPDATES DROP ya COALESCE ho sakte hain (jaise presence/
 *       typing/prices), ya HAR message deliver hona ZAROORI hai
 *       (jaise chat)?"
 * 
 * INTERVIEWER:
 *   "Achha, socho ek CHAT APP hai - latency ~1 second chahiye,
 *    BIDIRECTIONAL hai (dono taraf messages), scale medium-large
 *    hai, messages DURABLE hone chahiye, ordering zaroori hai."
 * 
 * CANDIDATE:
 *   "Perfect. Is information se clear hai:
 *      - Bidirectional + frequent messages -> WEBSOCKETS strong fit
 *      - Durability + ordering -> SEQUENCE IDs + durable log chahiye
 *    Lekin main pehle SAARE 4 APPROACHES explain karta hoon, taaki
 *    comparison clear ho, phir SPECIFIC recommendation dunga."
*/


/**
 * ====================================================================
 * ROUND 3: FOUR APPROACHES - Spectrum Samjho
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "4 common approaches kaunse hain real-time updates ke liye?"
 * 
 * CANDIDATE:
 *   "Yeh EK SPECTRUM banate hain - SIMPLE se SOPHISTICATED tak:
 * 
 *     DIAGRAM (Spectrum):
 * 
 *       [Short Polling] --> [Long Polling] --> [SSE] --> [WebSockets]
 *         Simplest                                        Most Complex
 *         Client-initiated                    Persistent    Bidirectional
 *         repeated ask                        server push   channel
 * 
 *    1. SHORT POLLING: Client baar baar poochta hai 'kuch naya hai?'
 *       Simple, lekin requests WASTE ho sakti hain.
 * 
 *    2. LONG POLLING: Server request ko OPEN rakhta hai jab tak
 *       DATA na aaye ya TIMEOUT na ho. Zyada EFFICIENT, phir bhi
 *       HTTP hai.
 * 
 *    3. SSE (Server-Sent Events): Server PERSISTENT one-way stream
 *       pe updates PUSH karta hai.
 * 
 *    4. WEBSOCKETS: FULL two-way communication EK persistent
 *       connection pe."
*/


/**
 * ====================================================================
 * ROUND 4: SHORT POLLING - Deep Dive
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Short polling kaise kaam karta hai?"
 * 
 * CANDIDATE:
 *   "Client EK TIMER set karta hai (jaise HAR 5 second), aur HAR
 *    baar HTTP request bhejta hai. Server check karta hai KUCH NAYA
 *    hai ya nahi, aur RESPOND karta hai UPDATES ya EMPTY response
 *    ke saath.
 * 
 *     DIAGRAM:
 * 
 *       Client --Any updates?--> Server --No--> [Wait 5 sec]
 *       Client --Any updates?--> Server --No--> [Wait 5 sec]
 *       Client --Any updates?--> Server --Yes, here's data!-->
 * 
 *    YEH approach PREDICTABLE aur EASY hai, lekin JAB MANY CLIENTS
 *    FREQUENTLY poll karte hain, SCALE nahi hota."
 * 
 * INTERVIEWER:
 *   "Iska fundamental trade-off kya hai?"
 * 
 * CANDIDATE:
 *   "LATENCY vs RESOURCE CONSUMPTION ka UNCOMFORTABLE choice hai:
 * 
 *     Aspect            | Poll Every 1 Sec           | Poll Every 30 Sec
 *     ------------------|----------------------------|--------------------------
 *     Latency           | Fast updates               | 30 sec tak delay
 *     Request Volume    | 3,600 req/hour per client  | 120 req/hour per client
 *     Server Load       | Constant heavy load        | Manageable
 *     Battery           | High drain (mobile)        | Battery-friendly
 * 
 *    ZYADA FREQUENTLY poll karo = FAST updates lekin BHARI server
 *    load. KAM FREQUENTLY poll karo = RESOURCE SAVE, lekin USER ko
 *    ZYADA WAIT karna padta hai. KOI MIDDLE GROUND nahi hai jo is
 *    trade-off se BACH sake.
 * 
 *    NUMBERS bhi bata deta hoon: 5-second polling interval + 1
 *    MILLION active clients = roughly 200,000 REQUESTS PER SECOND -
 *    aur INMEIN se ZYADATAR EMPTY responses hote hain! Yeh BAHUT
 *    BADI infrastructure cost hai kuch bhi na dene ke liye."
 * 
 * INTERVIEWER:
 *   "Fayde aur nuksaan batao."
 * 
 * CANDIDATE:
 *   "FAYDE:
 *      - IMPLEMENT karna SIMPLE hai - standard HTTP + timer
 *      - HAR browser/client mein WORK karta hai
 *      - STATELESS server-side, HORIZONTAL SCALING easy
 *      - DEBUG karna easy hai - har request-response independent hai
 * 
 *    NUKSAAN:
 *      - ZYADATAR requests KUCH NAHI return karti - bandwidth WASTE
 *      - Updates average mein HALF POLLING INTERVAL tak delayed
 *      - SCALE pe, MANY frequent-polling clients = HIGH server load
 *      - MOBILE BATTERY drain constant network activity se"
 * 
 * INTERVIEWER:
 *   "Kab use karoge, kab nahi?"
 * 
 * CANDIDATE:
 *   "GOOD FIT: Email inbox check, dashboard refresh, background
 *    sync, status page monitoring - jahan UPDATES INFREQUENT hain
 *    aur FEW SECONDS ka delay OK hai.
 * 
 *    POOR FIT: Chat, live location tracking, trading platforms,
 *    multiplayer games - jahan LATENCY TIGHT hai ya UPDATES
 *    CONTINUOUS hain."
*/


/**
 * ====================================================================
 * ROUND 5: LONG POLLING - Deep Dive
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Long polling short polling se KAISE DIFFERENT hai?"
 * 
 * CANDIDATE:
 *   "BASIC IDEA change ho jati hai - server 'KUCH NAHI HAI' bolne
 *    ki jagah, CONNECTION OPEN rakhta hai JAB TAK USKE PAAS KUCH
 *    BHEJNE KO NA HO.
 * 
 *     DIAGRAM:
 * 
 *       Client --Any updates? (Request 1)--> Server
 *                                         |
 *                               [No data yet, HOLD connection...]
 *                                         |
 *                               [Still waiting...]
 *                                         |
 *                               [New data arrives!]
 *                                         |
 *                                         v
 *                               Here is your update! (Response)
 *                                         |
 *                               Client processes update
 *                                         |
 *       Client --Any updates? (Request 2)--> Server (naya loop shuru)
 * 
 *    INSIGHT yeh hai: Client BAAR BAAR NAHI POOCHTA - EK BAAR
 *    POOCHTA HAI aur JAWAB KA WAIT karta hai. Server SIRF TAB
 *    RESPOND karta hai jab kuch MEANINGFUL ho."
 * 
 * INTERVIEWER:
 *   "Server-side sabse bada challenge kya hai?"
 * 
 * CANDIDATE:
 *   "Server ko WAIT karte waqt EK THREAD-PER-CONNECTION mein LOOP
 *    mein BUSY-WAIT NAHI karna chahiye - yeh CPU WASTE karega aur
 *    CONNECTION COUNT limit kar dega.
 * 
 *    PRODUCTION implementations EVENT-DRIVEN architectures aur
 *    PUB/SUB use karte hain:
 * 
 *     DIAGRAM:
 * 
 *       [Data Producer] --Publish--> [Topic: updates]
 *                                           |
 *                               -----------------------
 *                               |          |           |
 *                               v          v           v
 *                         [Notify]    [Notify]    [Notify]
 *                               |          |           |
 *                         Handler 1   Handler 2   Handler 3
 *                         (waiting)   (waiting)   (waiting)
 *                               |          |           |
 *                               v          v           v
 *                         Client 1    Client 2    Client 3
 * 
 *    Request AATE HI, HANDLER TOPIC mein INTEREST REGISTER karta hai
 *    aur CONTROL YIELD kar deta hai - THREAD BLOCK NAHI hota. NAYA
 *    update PUBLISH hote hi RELEVANT waiting handlers WAKE UP hote
 *    hain aur RESPOND karte hain.
 * 
 *    NOTE: Pub/Sub EPHEMERAL hai - agar CLIENTS ko DISCONNECT ke
 *    baad REPLAY chahiye, toh DURABLE LOG, STREAM, ya DATABASE-
 *    BACKED CURSOR use karna padega."
 * 
 * INTERVIEWER:
 *   "Fayde aur nuksaan?"
 * 
 * CANDIDATE:
 *   "FAYDE:
 *      - LOW-LATENCY delivery bina constant empty polling ke
 *      - Short polling se BAHUT KAM wasted requests
 *      - FIREWALLS/PROXIES ke through kaam karta hai (standard HTTP)
 *      - WebSockets se SIMPLE hai implement karna
 *      - Standard LOAD BALANCERS pe chal jata hai (idle timeout
 *        poll-hold-time se ZYADA tune karna hoga)
 * 
 *    NUKSAAN:
 *      - HAR waiting client EK server connection HOLD karta hai
 *        (resources consume)
 *      - ASYNC server frameworks chahiye THOUSANDS of concurrent
 *        connections efficiently handle karne ke liye
 *      - HAR response ke baad NAYI request establish karni padti hai
 *        - extra latency
 *      - TIMEOUT handling careful chahiye, especially UNSTABLE
 *        mobile connections ke liye
 *      - MESSAGE ORDERING tricky ho sakti hai jab responses aur new
 *        requests OVERLAP karein
 *      - CDN HELP NAHI karte - held-open responses DYNAMIC aur
 *        UNCACHEABLE hain"
 * 
 * INTERVIEWER:
 *   "Kab shine karta hai?"
 * 
 * CANDIDATE:
 *   "Notification systems, activity feeds, MODERATE-SCALE chat,
 *    progress updates - jahan REAL-TIME delivery chahiye lekin
 *    CLIENT ko FREQUENT messages BHEJNE ki zaroorat NAHI hai.
 *    WebSockets ki COMPLEXITY se pehle EK REASONABLE INTERMEDIATE
 *    step hai."
*/

/**
 * ====================================================================
 * ROUND 6: SERVER-SENT EVENTS (SSE) - Deep Dive
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "SSE kaise different hai in dono se?"
 * 
 * CANDIDATE:
 *   "SSE mein CLIENT BAAR BAAR NAHI POOCHTA. WOH EK SINGLE
 *    LONG-LIVED connection OPEN karta hai, aur SERVER JAB BHI KUCH
 *    CHANGE HOTA HAI, USI CONNECTION PE DATA PUSH KARTA HAI.
 * 
 *    Yeh CLOSER hai HUMARI INTUITIVE SOCH ke - server AAPKO NOTIFY
 *    karta hai jab kuch hota hai, aapko BAAR BAAR POOCHNA NAHI
 *    padta.
 * 
 *     DIAGRAM:
 * 
 *       Client --Open SSE connection--> Server
 *                                           |
 *                         [Connection STAYS OPEN]
 *                                           |
 *                               Price changed:
 *                               event: price-update
 *                               data: {symbol: AAPL, price: 150.25}
 *                                           |
 *                               New trade executed:
 *                               event: trade
 *                               data: {symbol: AAPL, shares: 100}
 *                                           |
 *                               Price changed AGAIN:
 *                               event: price-update
 *                               data: {symbol: AAPL, price: 150.50}
 *                                           |
 *                         [SAME connection - Multiple Events]
 * 
 *    CLIENT browser ki BUILT-IN 'EventSource' API use karta hai.
 *    Server SPECIAL content-type (text/event-stream) ke saath
 *    RESPOND karta hai, aur CONNECTION OPEN RAKHTA HAI."
 * 
 * INTERVIEWER:
 *   "SSE ka event format samjhao."
 * 
 * CANDIDATE:
 *   "SSE ek SIMPLE TEXT-BASED protocol use karta hai:
 * 
 *     ANATOMY OF SSE EVENT:
 * 
 *       event: price-update
 *       id: 12345
 *       data: {symbol: AAPL, price: 150.25}
 * 
 *     Field    | Purpose
 *     -----------|------------------------------------------
 *     event      | Event TYPE naam deta hai, client HANDLER route karta hai
 *     id         | UNIQUE identifier, DISCONNECT ke baad RESUME ke liye
 *     data       | ACTUAL payload, usually JSON
 *     retry      | Client ko batata hai KITNI DER wait kare RECONNECT se pehle"
 * 
 * INTERVIEWER:
 *   "SSE ka AUTOMATIC RECONNECTION kaise kaam karta hai?"
 * 
 * CANDIDATE:
 *   "BROWSER khud CONNECTION drop hone par RECONNECT karne ki
 *    koshish karta hai:
 * 
 *     DIAGRAM:
 * 
 *       [event id: 100] --> [event id: 101] --> [event id: 102]
 *                                           |
 *                               [Connection drops!]
 *                                           |
 *                               [Wait retry interval...]
 *                                           |
 *       Client --Reconnect, Last-Event-ID: 102--> Server
 *                                           |
 *                               [Resume from ID 102]
 *                                           |
 *       [event id: 103] --> [event id: 104] --> ...
 * 
 *    BROWSER 'Last-Event-ID' HEADER bhejta hai RECONNECT karte
 *    waqt, jisse SERVER RESUME kar sake AGAR server ke paas
 *    REPLAYABLE event log ho. Yeh USEFUL RESUME mechanism hai,
 *    lekin YEH APNE AAP mein DURABLE DELIVERY NAHI hai - server
 *    ko RECENT EVENTS store karne, GAPS handle karne, aur
 *    DEDUPLICATE karne ki zimmedari khud LENI PADEGI."
 * 
 * INTERVIEWER:
 *   "Fayde aur nuksaan?"
 * 
 * CANDIDATE:
 *   "FAYDE:
 *      - NATIVE browser support (EventSource API) - MINIMAL client
 *        code
 *      - AUTOMATIC reconnect + Last-Event-ID support
 *      - TEXT-based format = EASY debugging
 *      - Load balancers/proxies pe kaam karta hai (streaming
 *        timeouts/buffering configure karne ke baad)
 *      - WebSockets se LIGHTER weight hai
 * 
 *    NUKSAAN:
 *      - ONE-WAY ONLY - server push kar sakta hai, client NAHI
 *      - HTTP/1.1 par browsers PER-DOMAIN connections ~6 tak CAP
 *        karte hain, isliye MULTIPLE SSE streams EK PAGE PE doosre
 *        requests ko STARVE kar sakte hain (HTTP/2 multiplexing
 *        isse LARGELY solve karta hai)
 *      - Native EventSource API CUSTOM HEADERS set NAHI kar sakti,
 *        isliye AUTH usually COOKIES ya QUERY-PARAM TOKEN pe hoti
 *        hai, na ki Authorization header
 *      - TEXT-ONLY protocol - BINARY data ke liye Base64 encoding
 *        chahiye (overhead badhta hai)"
 * 
 * INTERVIEWER:
 *   "SSE vs Long Polling - key difference kya hai?"
 * 
 * CANDIDATE:
 *   "DIAGRAM:
 * 
 *       LONG POLLING:
 *         [Request 1] --> [Response + close] --> [Request 2] -->
 *         [Response + close] --> [Request 3] -->
 *         (HAR response ke baad NAYI request)
 * 
 *       SSE:
 *         [Single request] --> [Event 1] --> [Event 2] -->
 *         [Event 3] --> [Event 4]...
 *         (EK HI connection pe MULTIPLE updates)
 * 
 *    LONG POLLING HAR RESPONSE ke baad NAYI HTTP request banata
 *    hai. SSE EK CONNECTION pe MULTIPLE updates STREAM karta hai.
 *    Isse SSE FREQUENT server-to-client updates ke liye ZYADA
 *    EFFICIENT hai."
 * 
 * INTERVIEWER:
 *   "Kab SSE fit hota hai?"
 * 
 * CANDIDATE:
 *   "Jab data PRIMARILY server-to-client flow karta hai: STOCK
 *    PRICE tickers, LIVE SPORTS scores, NOTIFICATION streams,
 *    BUILD/JOB progress, SOCIAL MEDIA feeds.
 * 
 *    WEAK FIT hai for real-time CHAT, MULTIPLAYER games,
 *    COLLABORATIVE editing - jahan DONO SIDES FREQUENTLY messages
 *    bhejte hain, WEBSOCKETS BEHTAR hain.
 * 
 *    IMPORTANT POINT: Agar CLIENT KABHI KABHAR hi data bhejta hai
 *    (normal HTTP POST se KAAM chal jaye), SSE server push deta hai
 *    WEBSOCKETS ki COMPLEXITY ke BINA. BAHUT SI TEAMS WEBSOCKETS
 *    DEFAULT kar leti hain jab SSE HI KAAFI hota."
*/


/**
 * ====================================================================
 * ROUND 7: WEBSOCKETS - Deep Dive
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "WebSockets kaise kaam karte hain, aur kyun DIFFERENT hain?"
 * 
 * CANDIDATE:
 *   "WEBSOCKETS BIDIRECTIONAL communication dete hain - CLIENT aur
 *    SERVER DONO EK PERSISTENT connection pe MESSAGES bhej sakte
 *    hain.
 * 
 *    SSE server-to-client PUSH deta hai. WEBSOCKETS TWO-WAY channel
 *    dete hain - JAB DONO SIDES FREQUENTLY MESSAGES bhejte hain
 *    (chat, collaboration, games, trading)."
 * 
 * INTERVIEWER:
 *   "Protocol upgrade kaise hota hai?"
 * 
 * CANDIDATE:
 *   "Connection EK HTTP request ke roop mein SHURU hoti hai:
 * 
 *     DIAGRAM:
 * 
 *       PHASE 1 - HTTP Handshake:
 *         Client --GET /chat HTTP/1.1
 *                 Upgrade: websocket
 *                 Sec-WebSocket-Key: ...--> Server
 *                                              |
 *         Client <--HTTP/1.1 101 Switching Protocols
 *                   Sec-WebSocket-Accept: ...-- Server
 * 
 *       PHASE 2 - WebSocket Communication:
 *         Client <---WebSocket frame---> Server
 *         Client <---WebSocket frame---> Server
 *         (DONO TARAF SE KABHI BHI)
 * 
 *    CLIENT UPGRADE REQUEST bhejta hai. Agar SERVER AGREE karta
 *    hai, WOH '101 Switching Protocols' se RESPOND karta hai. Uske
 *    baad DONO SIDES HTTP ki jagah WEBSOCKET PROTOCOL BOLTE HAIN.
 *    Ab KOI REQUEST-RESPONSE pattern nahi hai - JISKO JAB CHAHE
 *    MESSAGE BHEJ SAKTA HAI."
 * 
 * INTERVIEWER:
 *   "Comparison table batao HTTP-based approaches vs WebSocket."
 * 
 * CANDIDATE:
 *   "Aspect        | HTTP-based Approaches        | WebSocket
 *    -----------------|---------------------------------|-------------------------------
 *    Initiation         | CLIENT HAR exchange initiate      | KOI BHI SIDE initiate kar sakta
 *                         karta hai                             hai
 *    Pattern              | Request/Response                    | KOI request/response nahi
 *    Connection              | Request/response ya HTTP stream      | Long-lived UPGRADED connection
 *    Overhead                  | HAR request pe headers                | KAM per-message overhead
 * 
 *    HTTP mein HAR EXCHANGE CLIENT se SHURU hoti hai. SSE mein bhi
 *    CLIENT hi connection OPEN karta hai. WEBSOCKETS mein SERVER
 *    BHI KABHI BHI CLIENT ko message BHEJ SAKTA HAI - isse TYPING
 *    INDICATORS, PRESENCE UPDATES, REAL-TIME collaboration jaise
 *    patterns POSSIBLE hote hain."
 * 
 * INTERVIEWER:
 *   "WebSockets ka scaling challenge kya hai?"
 * 
 * CANDIDATE:
 *   "STATELESS HTTP se ALAG, HAR CLIENT EK SPECIFIC SERVER se
 *    PERSISTENT connection maintain karta hai. HAR OPEN CONNECTION
 *    EK SOCKET (file descriptor) + SEND/RECEIVE buffers hold karta
 *    hai - isliye MEMORY aur OS ka FILE-DESCRIPTOR LIMIT ceiling
 *    set karte hain, CPU nahi.
 * 
 *    DEFAULT ULIMIT (1024 file descriptors per process) RAISE
 *    karna padta hai. REASONABLE RULE-OF-THUMB: TENS OF THOUSANDS
 *    idle connections per modern server node. TUNED event-driven
 *    setups mein HUNDREDS OF THOUSANDS se ~1 MILLION tak REACH kar
 *    sakte hain EK BOX pe (active message traffic isse KAM karta
 *    hai).
 * 
 *    INTERVIEW TAKEAWAY: WEBSOCKET TIER ko CONNECTION COUNT se
 *    SIZE karo, request rate se NAHI. Connection total badhne pe
 *    NODES add karo."
 * 
 * INTERVIEWER:
 *   "Agar Alice, Server 1 pe connected hai, aur Bob, Server 2 pe -
 *    Alice, Bob ko message kaise bhejegi?"
 * 
 * CANDIDATE:
 *   "COMMON SOLUTION hai EK MESSAGE BUS:
 * 
 *     DIAGRAM:
 * 
 *       [Client A - Alice] --> [Load Balancer] --> [Server 1]
 *                                                        |
 *                                               Subscribe & Publish
 *                                                        |
 *                                                        v
 *                                           [Message Bus - Redis/Kafka]
 *                                                        |
 *                                               Subscribe & Publish
 *                                                        |
 *                                                        v
 *                                                   [Server 2]
 *                                                        |
 *                                           [Client B - Bob]
 * 
 *    JAB ALICE MESSAGE bhejti hai, SERVER 1 use SHARED BUS pe
 *    PUBLISH karta hai, aur BUS use BOB ke connection wale server
 *    tak DELIVER karta hai. DO TAREEKE hain:
 * 
 *    1. BROADCAST: HAR server SAME topic subscribe karta hai, HAR
 *       message RECEIVE karta hai, aur JISKE PAAS wo user NAHI hai
 *       use DROP kar deta hai. EASY hai BUT WORK WASTE hota hai
 *       (har message HAR node ko wake karta hai).
 * 
 *    2. TARGETED ROUTING: DEKHO Bob KAUNSE SERVER pe hai, SIRF USI
 *       server ke channel pe publish karo. YEH BEHTAR SCALE karta
 *       hai lekin CONNECTION REGISTRY chahiye."
 * 
 * INTERVIEWER:
 *   "Connection registry kaise design karoge?"
 * 
 * CANDIDATE:
 *   "EK REGISTRY (jaise REDIS mein) maintain karunga jo HAR USER
 *    ko UNKE CONNECTED SERVER se MAP karta hai:
 * 
 *     DIAGRAM:
 * 
 *       Connection State (Redis):
 *         user:alice -> server1
 *         user:bob   -> server2
 *         user:carol -> server1
 * 
 *       MESSAGE ROUTING:
 *         Message for Bob
 *               |
 *         1. Lookup Bob's server (Redis se)
 *               |
 *         2. Publish to server2 channel
 *               |
 *         3. Server 2 --> Forward to Bob (Client B)
 * 
 *    HANDLE KARNA PADEGA: SERVER CRASH hone par, CLIENTS DIFFERENT
 *    servers pe RECONNECT karne par, aur REGISTRY out-of-sync hone
 *    par. HEARTBEATS, TTLs, aur DISCONNECT pe CLEANUP use karunga
 *    taaki STALE registry entries kisi WRONG server pe message
 *    route na karein."
 * 
 * INTERVIEWER:
 *   "Fayde aur nuksaan?"
 * 
 * CANDIDATE:
 *   "FAYDE:
 *      - TRUE BIDIRECTIONAL communication, KOI BHI SIDE KABHI BHI
 *        SEND kar sakti hai
 *      - LOW LATENCY, KAM per-message overhead
 *      - EFFICIENT binary framing
 *      - HIGH-FREQUENCY interactive apps ke liye acha fit
 * 
 *    NUKSAAN:
 *      - HTTP-based approaches se ZYADA COMPLEX implement/operate
 *        karna
 *      - SCALING ke liye MESSAGE BUS jaisa ADDITIONAL infra chahiye
 *      - STATEFUL connections STATELESS HTTP se INHERENTLY HARDER
 *        hain manage karna
 *      - KUCH corporate proxies/firewalls WEBSOCKET traffic BLOCK
 *        karte hain
 *      - BROWSER WebSocket API AUTOMATICALLY reconnect NAHI karta -
 *        KHUD IMPLEMENT karna padta hai
 *      - DEBUG karna HARDER hai - individual request/response
 *        INSPECT nahi kar sakte"
 * 
 * INTERVIEWER:
 *   "Kab worth hai complexity?"
 * 
 * CANDIDATE:
 *   "Jab BIDIRECTIONAL, HIGH-FREQUENCY communication chahiye:
 *    CHAT apps, MULTIPLAYER games, COLLABORATIVE editing,
 *    INTERACTIVE trading - kyunki DONO SIDES CONSTANTLY messages
 *    bhejte hain.
 * 
 *    NEWS feeds, notifications, live scores, progress tracking ke
 *    liye - FLOW MOSTLY ONE DIRECTION hai, isliye SSE USUALLY
 *    SIMPLER aur SUFFICIENT hai."
*/


/**
 * ====================================================================
 * ROUND 8: COMPARISON TABLE + DECISION TREE
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Chaaro approaches side-by-side compare karo."
 * 
 * CANDIDATE:
 *   "Aspect          | Short Poll    | Long Poll         | SSE         | WebSocket
 *    -------------------|-----------------|----------------------|----------------|------------------
 *    Communication         | Client polling   | Client waits          | Server stream  | Bidirectional
 *    Latency                 | Interval pe      | Low (jab request wait  | Low            | Low
 *                               depend             kar rahi ho)
 *    Request Overhead           | HAR poll EK req    | Response ke baad EK  | SINGLE stream  | SINGLE connection
 *                                                      req
 *    Server Complexity            | Simplest           | Moderate               | Moderate       | Highest
 *    Scaling Complexity              | Easy (stateless)   | Medium                | Medium         | Hard (stateful)
 *    Reconnection                      | Next poll           | Manual loop            | Browser auto   | Manual/library
 *    Firewall Friendly                    | Yes                | Yes                    | Yes            | Sometimes blocked
 *    Binary Data                            | Via encoding       | Via encoding            | Via encoding   | Native support"
 * 
 * INTERVIEWER:
 *   "Decision tree batao - kaise choose karoge?"
 * 
 * CANDIDATE:
 *   "MAIN QUESTIONS yeh hain:
 * 
 *     DIAGRAM (Decision Flow):
 * 
 *       [Application ki need kya hai?]
 *               |
 *       Seconds ka delay OK hai?
 *         YES -----------------> Large client count?
 *         |                         NO --> SHORT POLLING
 *         |                         YES --> (aage jaao, requests
 *         |                                  cost expensive hongi)
 *         NO
 *         |
 *         v
 *       Client FREQUENT messages bhejta hai?
 *         YES ---------------------------------> WEBSOCKETS
 *         NO
 *         |
 *         v
 *       Server-to-client STREAM chahiye?
 *         YES ------> SSE (ya HTTP request loop kaam kare toh
 *                      LONG POLLING)
 *         NO -------> SSE + HTTP POST (occasional client sends)
 * 
 *    KEY QUESTIONS:
 *      1. Kitni LATENCY acceptable hai, aur KIS SCALE pe? Agar FEW
 *         SECONDS ka delay theek hai aur CLIENT COUNT MODEST hai,
 *         SHORT POLLING kaafi ho sakta hai. LARGE CLIENT BASE ke
 *         saath, MOSTLY-EMPTY requests EXPENSIVE ho jati hain chahe
 *         LATENCY target LOOSE ho - toh LONG POLLING ya SSE BEHTAR
 *         fit hoti hai.
 *      2. Client FREQUENT upstream messages BHEJTA hai? YES toh
 *         WEBSOCKETS.
 *      3. Stream MOSTLY server-to-client hai? YES toh SSE, WEBSOCKETS
 *         se SIMPLER.
 *      4. REPLAY, ORDERING, ya DURABILITY chahiye? TRANSPORT AKELA
 *         yeh SOLVE nahi karta - SEQUENCE IDs, CURSORS, aur DURABLE
 *         EVENT SOURCE chahiye HOGI."
 * 
 * INTERVIEWER:
 *   "Public stock ticker mein toh THODA DELAY chal sakta hai - phir
 *    SSE kyun, short polling kyun nahi?"
 * 
 * CANDIDATE:
 *   "GOOD CATCH sir! Yeh SCALE BRANCH ki wajah se hai. Yeh feeds
 *    EK LARGE AUDIENCE ko serve karte hain, isliye MOSTLY-EMPTY
 *    SHORT-POLLING requests ka FLOOD EK SINGLE server-to-client
 *    STREAM se ZYADA COST karta hai - CHAHE LATENCY target LOOSE
 *    ho. Isliye SCALE bhi ek IMPORTANT FACTOR hai, sirf LATENCY
 *    NAHI."
*/


/**
 * ====================================================================
 * ROUND 9: PRODUCTION RELIABILITY - Interviewer Yahan Depth Test Karega
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Connections DROP ho jaate hain production mein. Kaise handle
 *    karoge reconnection?"
 * 
 * CANDIDATE:
 *   "EXPONENTIAL BACKOFF WITH JITTER use karunga:
 * 
 *     DIAGRAM:
 * 
 *       Attempt 1: wait 1s
 *       Attempt 2: wait 2s
 *       Attempt 3: wait 4s
 *       Attempt 4: wait 8s
 *       Attempt 5: wait 16s
 *       Attempt 6+: wait 30s (cap)
 * 
 *    SHORT DELAY (1 sec) se SHURU karo, HAR FAILURE pe DOUBLE karo,
 *    aur EK REASONABLE MAXIMUM (jaise 30 sec) pe CAP karo. RANDOM
 *    JITTER add karo taaki JAB MANY CLIENTS EK SAATH RECONNECT
 *    karein (jaise server restart ke baad), THUNDERING HERD PROBLEM
 *    NA HO. SUCCESSFUL connection ke baad DELAY COUNTER RESET karo."
 * 
 * INTERVIEWER:
 *   "Connection healthy hai ya dead, kaise pata karoge?"
 * 
 * CANDIDATE:
 *   "HEARTBEATS use karunga. NETWORK PERSPECTIVE se QUIET connection
 *    aur DEAD connection EK JAISE dikhte hain - ACTIVE PROBING ke
 *    bina PATA NAHI chalega.
 * 
 *     DIAGRAM:
 * 
 *       Client --PING--> Server --PONG--> Client
 *       [30 seconds silence...]
 *       Client --PING--> Server --PONG--> Client
 *       [30 more seconds...]
 *       [Server crashed!]
 *       Client --PING--> Server (NO PONG received...)
 *                               |
 *                         [Timeout! Reconnect.]
 * 
 *    HAR 30 SECONDS EK PING BHEJUNGA (reasonable default), aur EK
 *    TIMEOUT WINDOW ke ANDAR RESPONSE expect karunga. AGAR RESPONSE
 *    NAHI aaye, CONNECTION DEAD samjhunga aur RECONNECTION START
 *    karunga. Yeh SERVER CRASHES, NETWORK PARTITIONS, aur ZOMBIE
 *    connections CATCH karta hai jo TCP KEEPALIVES MISS kar sakte
 *    hain."
 * 
 * INTERVIEWER:
 *   "Messages OUT OF ORDER aa jayein ya DUPLICATE ho jayein, tab?"
 * 
 * CANDIDATE:
 *   "SEQUENCE NUMBERS ya UNIQUE IDs ASSIGN karunga HAR message ko.
 *    CLIENT LAST PROCESSED ID TRACK karega.
 * 
 *     DIAGRAM:
 * 
 *       Last seen: seq 102
 *               |
 *       Incoming message: seq 105
 *               |
 *       [GAP DETECTED - 103, 104 missing]
 *               |
 *       [Buffer until 103, 104 arrive]
 *               |
 *       Deliver IN ORDER: 103, 104, 105
 * 
 *    OUT-OF-ORDER messages ko BUFFER karunga GAPS FILL hone tak.
 *    DUPLICATES ko DETECT karke DISCARD karunga. SSE ke liye EVENT
 *    ID mechanism CLIENT ko batata hai KAHAN se RESUME karna hai,
 *    lekin SERVER ko REPLAYABLE event source RAKHNA HI PADEGA."
 * 
 * INTERVIEWER:
 *   "Client SLOW ho, keep up nahi kar paa raha - kya karoge?"
 * 
 * CANDIDATE:
 *   "SEND BUFFER SIZE MONITOR karunga. Agar CLIENT PEECHE reh jaye:
 * 
 *     DIAGRAM:
 * 
 *       Server generating 100 msgs/sec
 *               |
 *       Client processing 10 msgs/sec
 *               |
 *       [Send buffer growing...]
 *               |
 *       OPTIONS:
 *         1. Drop LOW-PRIORITY messages
 *         2. Aggregate/BATCH updates
 *         3. Disconnect SLOW client
 * 
 *    BEST CHOICE USE-CASE pe depend karta hai:
 *      - STOCK TICKER: INTERMEDIATE price updates DROP kar sakte
 *        hain, SIRF LATEST bhejo.
 *      - CHAT APP: HAR message deliver KARNA ZAROORI hai, isliye
 *        BAHUT SLOW clients ko DISCONNECT kar dena BEHTAR hai."
 * 
 * INTERVIEWER:
 *   "Long-lived connections mein AUTHENTICATION kaise handle
 *    karoge, jab token EXPIRE ho jaye connection ke live rehte
 *    hue?"
 * 
 * CANDIDATE:
 *   "PER-REQUEST HTTP AUTH SIMPLE hai - HAR request TOKEN carry
 *    karta hai. LONG-LIVED CONNECTIONS mein CLIENT EK BAAR
 *    CONNECT time pe AUTHENTICATE hota hai, lekin CONNECTION
 *    GHANTON tak OPEN reh sakta hai - TOKEN EXPIRE ho sakta hai
 *    STREAM LIVE rehte hue.
 * 
 *    COMMON PATTERN: HANDSHAKE ke TIME authenticate karo, phir
 *    SESSION VALID rakho. Jab TOKEN EXPIRY ke PAAS ho:
 *      - CLIENT REFRESHED token IN-BAND message ke through bheje, YA
 *      - SERVER CONNECTION CLOSE kar de, CLIENT FRESH token ke
 *        saath RECONNECT kare
 * 
 *    3 SECURITY concerns bhi explicit dhyaan mein rakhunga:
 * 
 *     Concern             | Risk                        | Mitigation
 *     -----------------------|--------------------------------|--------------------------------
 *     Transport encryption     | Tokens/payloads clear text mein | WSS/HTTPS use karo, PLAIN
 *                                                                    ws:// ya http:// KABHI NAHI
 *     Cross-site hijacking       | Malicious page WebSocket OPEN     | Origin header VALIDATE karo,
 *     (CSWSH)                       kare, BROWSER cookies attach       EXPLICIT token PREFER karo
 *                                     kar deta hai automatically         cookie-only auth se
 *     Token leakage                 | SSE EventSource Authorization    | Short-lived tokens, ya
 *                                     header set nahi kar sakta,          cookie SCOPED to streaming
 *                                     token QUERY STRING mein aata         endpoint
 *                                     hai (access logs mein land)"
 * 
 * INTERVIEWER:
 *   "Load balancing aur zero-downtime deploys ke saath WebSockets
 *    ka kya scene hai?"
 * 
 * CANDIDATE:
 *   "STATELESS HTTP mein LOAD BALANCER HAR request KISI BHI node
 *    ko bhej sakta hai. LONG-LIVED connections EK BAAR ESTABLISH
 *    hone ke baad MOVE NAHI hote - isse LOAD BALANCER layer ka
 *    BEHAVIOR badal jata hai.
 * 
 *    FIRST: LOAD BALANCER ko UPGRADE header samajhna PADEGA - YA
 *    TOH LAYER 7 pe operate karke UPGRADE PASS THROUGH kare, YA
 *    LAYER 4 pe PLAIN TCP PROXY ki tarah chale. KUCH PURANE PROXIES
 *    upgrade STRIP kar dete hain (common cause of 'WebSockets work
 *    locally but not in production'). IDLE TIMEOUTS bhi matter
 *    karte hain - HEARTBEATS ISI TIMER ko RESET karke KEEPALIVES ki
 *    tarah kaam karte hain.
 * 
 *    SECOND: DEPLOYS DISRUPTIVE hain STATELESS services se ALAG
 *    tarah se. EK NODE REPLACE karne se USKE SAARE connections EK
 *    SAATH DROP ho jate hain, aur WOH CLIENTS SECONDS mein EK SAATH
 *    RECONNECT karte hain.
 * 
 *    MITIGATIONS:
 *      - CONNECTION DRAINING (naye connections ACCEPT karna band
 *        karo, EXISTING ko GRACE PERIOD do finish/move hone ke liye)
 *      - STAGGERED ROLLOUTS (SIRF EK FRACTION connections EK BAAR
 *        mein drop hon)
 *      - CLIENT-SIDE EXPONENTIAL BACKOFF + JITTER (reconnections
 *        SPREAD OUT hon, EK SAATH BURST mein NA AAYE)"
*/

/** 
 * ====================================================================
 * ROUND 10: REAL-WORLD EXAMPLES - Interviewer Application Test Karega
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Slack, Discord, WhatsApp jaise apps kya use karte hain?"
 * 
 * CANDIDATE:
 *   "System          | Transport          | Why
 *    -------------------|-----------------------|----------------------------------
 *    Slack               | WebSocket             | Persistent connection - messages,
 *                                                     presence, typing - SAB DONO
 *                                                     directions mein FREQUENTLY
 *    Discord               | WebSocket gateway     | Ek GATEWAY connection, HEARTBEATS
 *                                                     for health, SEQUENCE NUMBERS +
 *                                                     RESUME PROTOCOL missed events
 *                                                     replay karne ke liye
 *    WhatsApp                | Persistent socket    | Long-lived connection (historically
 *                                                     customized XMPP over TCP);
 *                                                     WhatsApp Web WebSocket use karta hai
 *    Figma                     | WebSocket           | Multiplayer editing - CONTINUOUS
 *                                                     two-way sync document changes ka
 *    Coinbase/Binance             | WebSocket feeds    | Order-book/trade updates, MOSTLY
 *                                                     server-to-client, WebSocket
 *                                                     subscription control ke liye
 *    LLM chat UIs (ChatGPT)          | SSE               | Token-by-token streaming ONE-
 *                                                     DIRECTIONAL hai, isliye SSE
 *                                                     NATURAL fit hai"
 * 
 * INTERVIEWER:
 *   "Discord ka example interesting hai - kyun?"
 * 
 * CANDIDATE:
 *   "Discord EK CLEAN EXAMPLE hai HUMNE JO RELIABILITY MACHINERY
 *    discuss ki, uski PRODUCTION mein: WEBSOCKET LIVE STREAM carry
 *    karta hai, LEKIN HEARTBEATS DEAD connections DETECT karte
 *    hain, aur SEQUENCE-NUMBER + RESUME PROTOCOL MISSED events
 *    REPLAY karta hai - kyunki TRANSPORT AKELA DELIVERY guarantee
 *    NAHI karta."
 * 
 * INTERVIEWER:
 *   "LLM chat UIs SSE kyun use karte hain, WebSocket nahi?"
 * 
 * CANDIDATE:
 *   "Kyunki DATA SIRF EK DIRECTION mein FLOW karta hai - SERVER se
 *    CLIENT tak, TOKEN BY TOKEN. WEBSOCKET ki FULL BIDIRECTIONAL
 *    COMPLEXITY ki ZAROORAT HI NAHI hai. Yeh EK ACHA REMINDER hai
 *    ki DIRECTIONALITY hi TRANSPORT choice ka MAIN DRIVER hai."
*/


/**
 * ====================================================================
 * ROUND 11: BEYOND THESE FOUR - Related Technologies
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Agar mobile app BACKGROUND mein hai ya CLOSED hai, WebSocket
 *    ya SSE se update kaise dikhaoge?"
 * 
 * CANDIDATE:
 *   "IN SAB approaches mein se KOI KAAM NAHI KAREGA jab app
 *    BACKGROUNDED ya CLOSED hai, kyunki OS APP ko SUSPEND kar deta
 *    hai aur BATTERY BACHANE ke liye CONNECTIONS TEAR DOWN kar deta
 *    hai.
 * 
 *    In cases mein PLATFORM PUSH SERVICE chahiye: APNs (Apple Push
 *    Notification service) ya FCM (Firebase Cloud Messaging).
 * 
 *    APP SERVER MESSAGE, APNs/FCM ko hand karta hai, jo APNE
 *    CONNECTION ke through DEVICE ko WAKE karta hai. REAL APPS
 *    DONO MODELS COMBINE karte hain: APP FOREGROUND mein hai toh
 *    LIVE CONNECTION, BACKGROUND mein hai toh PUSH NOTIFICATIONS."
 * 
 * INTERVIEWER:
 *   "WebRTC kab use karoge?"
 * 
 * CANDIDATE:
 *   "Jab SABSE KAM LATENCY ya DIRECT PEER-TO-PEER MEDIA chahiye -
 *    jaise VIDEO CALLS, VOICE, SCREEN SHARING. WEBRTC BROWSERS ke
 *    BEECH DIRECT CONNECTION establish karta hai (RELAYS pe FALLBACK
 *    karta hai zaroorat pe), USUALLY UDP par - jo GUARANTEED
 *    DELIVERY ko SPEED ke liye TRADE karta hai.
 * 
 *    YEH SETUP karna MORE COMPLEX hai in chapter ke approaches se,
 *    aur USUALLY REAL-TIME AUDIO/VIDEO ke liye RESERVE hota hai,
 *    general DATA UPDATES ke liye NAHI."
 * 
*/

/**
 * ====================================================================
 * ROUND 12: CLOSING - Final Summary
 * ====================================================================
 * 
 * INTERVIEWER:
 *   "Theek hai, ab CHAT APP wapas leke aata hoon jo maine round 2
 *    mein diya tha - bidirectional, ~1sec latency, durable, ordered
 *    messages. Final design do."
 * 
 * CANDIDATE:
 *   "Sure sir. MERI FINAL DESIGN yeh hogi:
 * 
 *    1. TRANSPORT: WEBSOCKETS, kyunki COMMUNICATION BIDIRECTIONAL
 *       hai aur HIGH FREQUENCY hai (typing indicators, messages,
 *       read receipts).
 * 
 *    2. MESSAGE ROUTING: EK CONNECTION REGISTRY (Redis) MAINTAIN
 *       karunga jo har USER ko UNKE SERVER se MAP karta hai, aur
 *       MESSAGE BUS (Kafka/Redis Pub-Sub) use karunga CROSS-SERVER
 *       delivery ke liye - TARGETED ROUTING approach se, BROADCAST
 *       se NAHI.
 * 
 *    3. DURABILITY/ORDERING: HAR MESSAGE ko SEQUENCE ID dunga,
 *       DURABLE LOG/DATABASE mein STORE karunga taaki REPLAY POSSIBLE
 *       ho DISCONNECT ke baad.
 * 
 *    4. RELIABILITY: HEARTBEATS (30 sec) CONNECTION HEALTH ke liye,
 *       EXPONENTIAL BACKOFF + JITTER RECONNECTION ke liye.
 * 
 *    5. AUTH: HANDSHAKE pe AUTHENTICATE, ORIGIN header VALIDATE
 *       karunga (CSWSH se BACHNE ke liye), TOKEN REFRESH IN-BAND
 *       handle karunga.
 * 
 *    6. SCALING: CONNECTION COUNT ke basis pe SIZE karunga (file
 *       descriptor limits raise), CONNECTION DRAINING + STAGGERED
 *       ROLLOUTS zero-downtime deploys ke liye.
 * 
 *    7. MOBILE: JAB APP BACKGROUND mein ho, FCM/APNs push
 *       NOTIFICATIONS use karunga."
 * 
 * 
 * ====================================================================
 * RAPID FIRE REVISION (Last Minute Ke Liye)
 * ====================================================================
 * 
 * Q: "Real-time" ka matlab fixed nahi hai - kyun?
 * A: Context pe depend karta hai - editor ~100ms, chat ~1sec,
 *    tracking ~5sec, dashboard ~30sec. Target latency transport
 *    choose karti hai.
 * 
 * Q: 4 approaches kya hain, simple se complex order mein?
 * A: Short Polling -> Long Polling -> SSE -> WebSockets.
 * 
 * Q: Short polling ka fundamental trade-off?
 * A: Latency vs Resource - frequent poll = fast but heavy load;
 *    rare poll = light but slow updates. Koi middle ground nahi.
 * 
 * Q: Long polling short polling se kaise better hai?
 * A: Server connection HOLD karta hai jab tak data na ho, isliye
 *    WASTED empty requests bahut kam ho jaati hain.
 * 
 * Q: SSE ka core idea?
 * A: Server ek SINGLE persistent connection pe MULTIPLE events push
 *    karta hai - client ko baar baar poochna nahi padta.
 * 
 * Q: SSE ki sabse badi limitation?
 * A: ONE-WAY only - server client ko bhej sakta hai, client server
 *    ko NAHI (SSE ke through).
 * 
 * Q: WebSockets kab use karein?
 * A: BIDIRECTIONAL + HIGH-FREQUENCY communication chahiye ho -
 *    chat, games, collaborative editing, trading.
 * 
 * Q: WebSocket scaling kis cheez se limited hoti hai - CPU se?
 * A: NAHI - MEMORY aur FILE DESCRIPTOR LIMIT se. Connection count
 *    se size karo, request rate se nahi.
 * 
 * Q: Cross-server WebSocket messaging kaise solve karte hain?
 * A: Message Bus (Redis/Kafka) + Connection Registry (jo batata hai
 *    kaunsa user kaunse server pe hai) - TARGETED ROUTING best hai.
 * 
 * Q: Transport delivery guarantee deta hai kya?
 * A: NAHI - transport sirf CHANNEL hai. Ordering/replay/durability
 *    ke liye SEQUENCE IDs, DURABLE LOG, DEDUPLICATION chahiye alag
 *    se.
 * 
 * Q: CSWSH kya hai?
 * A: Cross-Site WebSocket Hijacking - malicious page WebSocket
 *    opens karta hai, browser cookies auto-attach kar deta hai.
 *    Fix: Origin header validate karo.
 * 
 * Q: Background/closed mobile app ko update kaise bhejein?
 * A: WebSocket/SSE kaam nahi karte (OS connection tear down karta
 *    hai). PLATFORM PUSH (APNs/FCM) use karo.
 * 
 * Q: LLM chat streaming (ChatGPT) SSE kyun use karta hai?
 * A: Kyunki data ONE-DIRECTIONAL hai (server-to-client token
 *    streaming), WebSocket ki bidirectional complexity ki zaroorat
 *    nahi.
 * 
 * ====================================================================
 *                     END OF INTERVIEW PREP NOTES
 *    (Practice bolke karo, whiteboard pe har approach ka diagram
 *     banane ki practice bhi karo - interview mein yehi flow
 *     follow karna hai)
 * ====================================================================
*/

In [ ]:
/*
=============================================================================
          REAL-TIME UPDATES IN SYSTEM DESIGN - COMPLETE NOTES
=============================================================================
Bhai, aaj hum samjhenge ki real-time updates kaise kaam karte hain.
Ye system design interviews mein bahut common topic hai.
Chal shuru karte hain!

=============================================================================
SECTION 1: REAL-TIME KYA HOTA HAI?
=============================================================================

Real-time ka matlab hai -> user ko turant (ya jaldi se jaldi) data dikhana
Bina page refresh kiye data update ho jaaye.

Examples:
  - WhatsApp pe message aate hi dikh jaata hai       -> ~1 sec
  - Google Docs mein saath milke edit karna           -> ~100 ms
  - Uber mein driver ki live location                 -> ~2-5 sec
  - Dashboard pe data refresh                         -> ~30 sec

IMPORTANT BAAT:
"Real-time" ek hi requirement nahi hai!
Har app ka latency target alag hota hai:
  - Collaborative editor  = 100ms chahiye
  - Chat app              = 1 second chahiye
  - Driver tracking       = 5 seconds chahiye
  - Dashboard             = 30 seconds chahiye

Ye target decide karta hai ki kaunsi technology use karni hai.


=============================================================================
SECTION 2: TRADITIONAL REQUEST/RESPONSE KYUN KAAM NAHI KARTA?
=============================================================================

Normal web request aise kaam karta hai:

  Client                          Server
    |                               |
    |--- "Mujhe data do" --------->|
    |<-- "Ye lo data" ------------|
    |                               |
    |   [CONNECTION KHATAM]         |
    |                               |

Problem kya hai?
  - Agar server pe kuch change ho jaaye, client ko pata hi nahi chalta
  - Client ko baar baar request karni padti hai -> "kuch naya hai?"
  - Ye bahut wasteful hai

Isliye real-time techniques chahiye!


=============================================================================
SECTION 3: 4 APPROACHES KA OVERVIEW
=============================================================================

Ye 4 approaches hain, simple se complex ki taraf:

  Simple +---------------------------------------------+ Complex
         |          |            |            |          |
      Short      Long         SSE        WebSocket
      Polling    Polling                  (WS)
         |          |            |            |
     Client      Client       Server      Bidirectional
     repeatedly  asks once,    pushes      dono side
     asks        server       data ek     se data
     "kuch naya?" holds till   stream      jaata hai
                 data aaye    mein

  Latency:    High ---------> Low
  Complexity: Low ----------> High


=============================================================================
SECTION 4: SHORT POLLING (Sabse Simple)
=============================================================================

KAISE KAAM KARTA HAI:
  Client baar baar request bhejta hai fixed interval pe.
  Jaise har 5 second pe -> "bhai kuch naya hai?"

  Client                          Server
    |                               |
    |--- "Kuch naya?" ------------>|
    |<-- "Nahi bhai" -------------|
    |                               |
    |   [5 sec wait]                |
    |                               |
    |--- "Kuch naya?" ------------>|
    |<-- "Nahi bhai" -------------|
    |                               |
    |   [5 sec wait]                |
    |                               |
    |--- "Kuch naya?" ------------>|
    |<-- "Haan ye lo data!" ------|
    |                               |

TRADE-OFF (Sabse important baat):

  Agar har 1 sec pe poll karo:
    + Updates jaldi milenge
    - 3600 requests/hr per client (bahut zyada!)
    - Server pe bahut load
    - Mobile ki battery khatam

  Agar har 30 sec pe poll karo:
    + Server ka load kam
    + Battery friendly
    - Updates mein 30 sec tak ka delay

  Ek simple calculation:
    1 million users, 5 sec interval = 200,000 requests/sec
    Aur in mein se bahut saari EMPTY response hongi!
    Ye paisa barbaad hai bhai.

FAAYDE (Pros):
  + Implement karna bahut easy hai
  + Har browser, har client support karta hai
  + Server stateless hai -> horizontal scaling easy
  + Debug karna easy (har request independent hai)

NUKSAAN (Cons):
  - Zyada tar requests khaali return karti hain
  - Average delay = polling interval / 2
  - Scale pe bahut expensive
  - Mobile battery drain

KAB USE KAREIN:
  Good for:  Email check, dashboard refresh, status page
  Bad for:   Chat, live tracking, trading, multiplayer games


=============================================================================
SECTION 5: LONG POLLING (Better version)
=============================================================================

SHORT POLLING vs LONG POLLING - Core Difference:

  Short Polling:
    Client: "Kuch naya?"
    Server: "Nahi"  [turant reply de deta hai, empty]
    Client: "Kuch naya?"  [phir se puchta hai]

  Long Polling:
    Client: "Kuch naya?"
    Server: "Ruk bhai, jab aayega tab bataunga" [HOLD karta hai]
    Server: [data aaya!] "Haan ye lo!" [ab reply karta hai]

KAISE KAAM KARTA HAI:

  Client                          Server         DataStore
    |                               |                |
    |--- "Kuch naya?" ------------>|                |
    |                               |                |
    |   [Server WAIT karta hai...]  |                |
    |   [koi data nahi hai...]      |                |
    |                               |                |
    |                    [Naya data aaya!] <---------|
    |                               |                |
    |<-- "Ye lo naya data!" ------|                |
    |                               |                |
    |   [Client process karta hai]  |                |
    |                               |                |
    |--- "Kuch naya?" ------------>|  [Naya request]|
    |                               |                |

SERVER SIDE CHALLENGE:
  Bhai, agar server ek thread baitha ke wait kare har client ke liye,
  toh 10000 clients = 10000 threads = server crash!

  Solution: ASYNC / EVENT-DRIVEN architecture use karo

  Data Producer
       |
       | Publish
       v
  [Pub/Sub or Message Bus]  (jaise Redis Pub/Sub)
       |        |        |
       | Notify | Notify | Notify
       v        v        v
  Handler1  Handler2  Handler3
  (waiting)  (waiting)  (waiting)
  for        for        for
  Client1    Client2    Client3

  Kya hota hai:
    1. Client request bhejta hai
    2. Handler register karta hai "mujhe is topic pe data chahiye"
    3. Handler WAIT karta hai (thread block nahi hota, yield karta hai)
    4. Jab data aata hai topic pe, handler wake up hota hai
    5. Handler client ko response bhejta hai

FAAYDE:
  + Low latency, khaali requests bahut kam
  + Standard HTTP hai -> firewall/proxy friendly
  + WebSockets se simple hai

NUKSAAN:
  - Har waiting client ek open connection hold karta hai
  - Async server framework chahiye (Node.js, Netty, etc.)
  - Har response ke baad naya request banana padta hai (small gap)
  - Timeout handling tricky hai mobile pe
  - CDN help nahi karta (held responses dynamic hain)

KAB USE KAREIN:
  Notifications, activity feeds, moderate-scale chat, progress updates


=============================================================================
SECTION 6: SERVER-SENT EVENTS (SSE)
=============================================================================

SSE ka concept:
  Client ek baar connection open karta hai.
  Server ussi connection pe baar baar data push karta hai.
  Client ko baar baar request nahi karni padti!

  Client                          Server
    |                               |
    |--- [Open SSE Connection] ---->|
    |                               |
    |<-- event: price-update -------|  [Server push]
    |    data: {price: 150.25}      |
    |                               |
    |<-- event: trade --------------|  [Server push]
    |    data: {shares: 100}        |
    |                               |
    |<-- event: price-update -------|  [Server push]
    |    data: {price: 150.50}      |
    |                               |
    |   [Connection stays OPEN]     |

KAISE KAAM KARTA HAI:
  1. Client browser ki EventSource API use karta hai
  2. Server response bhejta hai with content-type: text/event-stream
  3. Connection OPEN rehta hai
  4. Server jab chahe data push karta hai
  5. Browser automatically reconnect karta hai agar connection toote

SSE EVENT FORMAT (Text-based, simple!):

  event: price-update          <- event type (kis type ka event hai)
  id: 12345                    <- unique ID (reconnect pe resume ke liye)
  data: {symbol: AAPL, price: 150.25}  <- actual data (usually JSON)
  retry: 5000                  <- kitna wait kare reconnect pe (milliseconds)

  Field     | Kaam
  ----------|------------------------------------------
  event     | Event ka naam, client specific handler call kare
  id        | Resume ke liye unique identifier
  data      | Actual payload (JSON usually)
  retry     | Reconnect hone pe kitna wait kare

AUTOMATIC RECONNECTION (Bahut useful feature!):

  Client                          Server
    |                               |
    |--- [Connected] ------------->|
    |<-- event id:100 -------------|
    |<-- event id:101 -------------|
    |<-- event id:102 -------------|
    |                               |
    |   [CONNECTION DROP!]          |
    |   [Network issue / server restart]|
    |                               |
    |   [Browser wait karta hai retry time]|
    |                               |
    |--- Reconnect --------------->|
    |    Last-Event-ID: 102        |  <- "102 ke baad ka data do"
    |                               |
    |<-- event id:103 -------------|
    |<-- event id:104 -------------|

  BROWSER khud se reconnect karta hai!
  Last-Event-ID header bhejta hai -> server ko pata hai kahaan se resume kare

  Lekin dhyan rakho:
    - Server ko recent events STORE karna padta hai
    - Client ko deduplication karni padti hai
    - Ye DELIVERED nahi guarantee karta, sirf resume ka mechanism hai

SSE vs LONG POLLING - Visual Comparison:

  Long Polling:
    Request1 -> Response -> [CLOSE]
                              Request2 -> Response -> [CLOSE]
                                                       Request3 -> ...
    Har response ke baad naya request! Beech mein gap hai.

  SSE:
    Single Request
       |-> Event 1
       |-> Event 2
       |-> Event 3
       |-> Event 4...
    Ek hi connection, multiple events stream hote hain!

  SSE zyada efficient hai jab frequent updates aaye server se.

FAAYDE:
  + Browser built-in support (EventSource API)
  + Automatic reconnect + Last-Event-ID
  + Text format -> debug karna easy
  + HTTP load balancers ke saath kaam karta hai
  + WebSockets se lightweight hai

NUKSAAN:
  + SIRF ONE-WAY: Server -> Client (client server ko directly nahi bhej sakta)
  - HTTP/1.1 pe ~6 connections limit per domain (HTTP/2 mein problem nahi)
  - EventSource API custom headers set nahi kar sakti (auth tricky)
  - Text only protocol (binary ke liye Base64 encode karna padta hai)

KAB USE KAREIN:
  Stock tickers, live sports scores, notification streams, build progress,
  social media feeds, LLM streaming (ChatGPT jaisa token-by-token)


=============================================================================
SECTION 7: WEBSOCKETS (Full Power!)
=============================================================================

WEBSOCKET kya hai:
  Bidirectional communication - dono side se data ja sakta hai
  Client server ko bhej sakta hai, server client ko bhej sakta hai
  Ek single persistent connection pe!

  Client                          Server
    |                               |
    |--- [HTTP Upgrade Request] --->|  <- "Bhai, HTTP chhod ke
    |    Upgrade: websocket         |     WebSocket pe aa jaao"
    |                               |
    |<-- 101 Switching Protocols ---|  <- "Theek hai, switch karte hain"
    |                               |
    |   [Ab WebSocket connection hai]|
    |                               |
    |<-- User 2 is typing... ------|  <- Server push
    |--- "Hello!" ---------------->|  <- Client send
    |<-- Delivery confirmation ----|  <- Server push
    |--- "Kaisa hai?" ------------>|  <- Client send
    |<-- "Acha hoon!" ------------|  <- Server push (dusre user ka)
    |                               |

PROTOCOL UPGRADE (Handshake):

  Phase 1: HTTP Request
    GET /chat HTTP/1.1
    Upgrade: websocket
    Sec-WebSocket-Key: x3JJHMbDL1EzLkh9GBhXDw==

  Phase 2: Server Response
    HTTP/1.1 101 Switching Protocols
    Sec-WebSocket-Accept: HSmrc0sMlYUkAGmm5OPpG2HaGWk=

  Phase 3: Ab dono WebSocket frames mein baat karte hain
    [Frame] [Frame] [Frame] [Frame]  <- Dono taraf se

HTTP vs WEBSOCKET - Key Differences:

  Aspect          | HTTP-Based          | WebSocket
  ----------------|---------------------|------------------
  Initiation      | Client start kare   | Dono side start kar sakte
  Pattern         | Request/Response    | Koi pattern nahi
  Connection      | Short-lived         | Long-lived
  Overhead        | Har request pe headers | Per-message overhead kam

THE SCALING CHALLENGE (Bahut important!):

  Problem: WebSockets = STATEFUL connections
  Har client ek specific server se connected hai.

  1 Million clients chahiye? -> Bahut saare servers chahiye

  Har open connection hold karta hai:
    - Socket (file descriptor)
    - Send buffer
    - Receive buffer

  Default ulimit = 1024 file descriptors per process
  Ye BADHANA padta hai! (ulimit -n 65535 etc.)

  Ek modern server node:
    - Idle connections: tens of thousands
    - Event-driven stack: hundreds of thousands to ~1 million
    - Active traffic: kam ho jaata hai

  INTERVIEW TIP:
    "WebSockets ko CONNECTION COUNT se size karte hain,
     REQUEST RATE se nahi!"

CROSS-SERVER MESSAGING (Sabse tricky part):

  Problem: Alice Server 1 pe hai, Bob Server 2 pe hai.
  Alice ko message bhejna hai Bob ko. Kaise?

  Client A (Alice)          Client B (Bob)
       |                         |
       |                         |
   [Server 1]              [Server 2]
       |                         |
       |     ??? Kaise ???       |
       |                         |
       +------[Message Bus]------+
              (Redis Pub/Sub
               ya Kafka)

  Solution: MESSAGE BUS

  Option 1 - BROADCAST (Simple but wasteful):
    Server 1: "Ye message hai Bob ke liye"
    [Message Bus -> Sabhi servers ko bhejo]
    Server 1: "Bob yahan nahi hai, DROP"
    Server 2: "Bob yahan hai, deliver karo!"

    Problem: Har message har server pe jaata hai -> waste

  Option 2 - TARGETED ROUTING (Efficient but complex):
    1. Lookup karo -> Bob kis server pe hai
    2. Sirf usi server ke channel pe publish karo

    Message for Bob
         |
    1. Lookup Bob's server
         |
    [Connection State - Redis]
    user:alice -> server1
    user:bob   -> server2    <- "Bob server2 pe hai"
    user:carol -> server1
         |
    2. Publish to server2 channel
         |
    Server 2 -> Forward to Bob

CONNECTION STATE MANAGEMENT:

  Har user ka mapping store karna padta hai:
    Redis mein:
      user:alice -> server1
      user:bob   -> server2
      user:carol -> server1

  Handle karna padta hai:
    - Server crash ho jaaye toh?
    - Client reconnect kare different server pe toh?
    - Registry out of sync ho jaaye toh?

  Solutions:
    - Heartbeats (regularly check karo connection alive hai)
    - TTL (entries auto-expire ho jaayein)
    - Cleanup on disconnect (disconnect pe registry se hatao)

FAAYDE:
  + TRUE bidirectional - dono side se kuch bhi bhejo
  + Low latency + kam per-message overhead
  + Binary framing support (efficient)
  + High-frequency interactive apps ke liye perfect

NUKSAAN:
  + Bahut complex hai implement aur operate karna
  + Extra infrastructure chahiye (message bus)
  + Stateful connections hard to manage
  + Kuch corporate proxies/firewalls block karte hain
  + Browser API auto-reconnect nahi karta (khud karna padta hai)
  + Debug karna mushkil

KAB USE KAREIN:
  Chat apps, multiplayer games, collaborative editing, trading platforms


=============================================================================
SECTION 8: COMPARISON TABLE (Yaad Rakhne Wali!)
=============================================================================

  Aspect          | Short     | Long      | SSE       | WebSocket
                  | Polling   | Polling   |           |
  ----------------|-----------|-----------|-----------|------------
  Communication   | Client    | Client    | Server->  | Bidirectional
                  | initiated | initiated | Client    |
  Latency         | Depends   | Low       | Low       | Low
                  | on interval|          |           |
  Request overhead| Har poll  | Har response| Single  | Single
                  | pe request| pe request | stream   | connection
  Server          | Simplest  | Moderate  | Moderate  | Highest
  complexity      |           |           |           |
  Scaling         | Easy      | Medium    | Medium    | HARD
  complexity      | (stateless)|          |           | (stateful)
  Reconnection    | Next poll | Manual    | Browser   | Manual/
                  |           | loop      | built-in  | library
  Browser support | Universal | Universal | Modern    | Modern
  Firewall        | Yes       | Yes       | Yes       | Sometimes
  friendly        |           |           |           | blocked
  Binary data     | Encoding  | Encoding  | Encoding  | Native!


=============================================================================
SECTION 9: DECISION TREE (Interview mein use karo!)
=============================================================================

  Start: App ko kya chahiye?
    |
    v
  "Seconds of delay OK?" (kuch seconds ka delay theek hai?)
    |
    +-- HAAN -> "Large client count?"
    |             |
    |             +-- HAAN -> SSE (scale pe efficient)
    |             |
    |             +-- NAHI -> Short Polling (simple hai)
    |
    +-- NAHI (low latency chahiye)
         |
         v
       "Client frequent messages bhejta hai?"
         |
         +-- HAAN -> WEBSOCKETS
         |
         +-- NAHI
              |
              v
            "Server-to-client stream chahiye?"
              |
              +-- HAAN -> SSE
              |
              +-- NAHI -> "HTTP request loop kaam karega?"
                             |
                             +-- HAAN -> Long Polling
                             |
                             +-- NAHI -> SSE + HTTP POST


=============================================================================
SECTION 10: USE CASE RECOMMENDATIONS
=============================================================================

  Use Case              | Recommended     | Kyun?
  ----------------------|-----------------|---------------------------
  Email inbox           | Short Polling   | Har minute check -> simple
  Dashboard metrics     | Short/SSE       | Freshness requirement pe depend
  Notification system   | SSE             | Server push, built-in reconnect
  Activity feed         | SSE             | One-direction continuous stream
  Stock price ticker    | SSE             | Frequent server->client updates
  Live sports scores    | SSE             | Real-time server push
  Build/CI progress     | SSE             | One-way progress updates
  Basic chat            | Long Poll / SSE | Moderate scale ke liye enough
  Real-time chat        | WebSocket       | Typing indicators, read receipts
  Multiplayer games     | WebSocket       | Low-latency bidirectional
  Collaborative editing | WebSocket       | Continuous bidirectional sync
  Trading platform      | WebSocket       | Bidirectional low-latency

NOTE: Stock ticker aur live scores SSE pe kyun?
  - Latency target loose hai (seconds OK)
  - LEKIN audience bahut bada hai (millions)
  - Short polling = millions of empty requests = expensive
  - SSE = ek stream = efficient
  - Isliye scale ki wajah se SSE choose karte hain!


=============================================================================
SECTION 11: IMPLEMENTATION BEST PRACTICES
=============================================================================

Har real-time system mein ye 4 problems aati hain:

  +-------------------+  +-------------------+
  | Connection        |  | Health            |
  | Management        |  | Detection         |
  +-------------------+  +-------------------+
  +-------------------+  +-------------------+
  | Message           |  | Flow              |
  | Ordering          |  | Control           |
  +-------------------+  +-------------------+

-----------------------------
11a. RECONNECTION WITH EXPONENTIAL BACKOFF
-----------------------------

Connections TOOTENGI. Guaranteed.
  - Mobile user WiFi <-> Cellular switch karega
  - Server restart hogi deployment pe
  - Network equipment fail hoga

Strategy: Exponential Backoff with Jitter

  Attempt 1: wait 1 second
  Attempt 2: wait 2 seconds
  Attempt 3: wait 4 seconds
  Attempt 4: wait 8 seconds
  Attempt 5: wait 16 seconds
  Attempt 6+: wait 30 seconds (CAP - max wait)

  + Jitter = thoda random add karo (0-1 sec)
    Kyun? -> Agar 1000 clients ek saath disconnect ho,
    sab ek saath reconnect karengi = THUNDERING HERD
    Jitter se alag alag time pe reconnect karengi

  + Successful connection ke baad -> delay counter RESET karo

-----------------------------
11b. HEARTBEATS (Connection Health Check)
-----------------------------

Problem: Ek connection jo chup hai aur ek dead connection
         NETWORK ke liye SAME lagte hain!

Solution: Regularly PING-PONG karo

  Client                          Server
    |                               |
    |--- "Hello!" ---------------->|
    |<-- "Hi!" -------------------|
    |                               |
    |   [30 sec silence...]         |
    |                               |
    |--- PING ------------------->|
    |<-- PONG --------------------|  <- "Main zinda hoon"
    |                               |
    |   [30 sec silence...]         |
    |                               |
    |--- PING ------------------->|
    |   [No response...]           |  <- Server crash ho gaya!
    |   [Timeout!]                 |
    |   [RECONNECT!]               |

  Default: Har 30 sec pe PING, kuch sec mein PONG expect karo
  Agar PONG nahi aaya -> connection DEAD hai -> reconnect

-----------------------------
11c. MESSAGE ORDERING & DEDUPLICATION
-----------------------------

Problem: Messages galat order mein aa sakte hain
         Especially reconnect ke baad

  Incoming message: seq 105
  Last processed:   seq 102
  Kya karein?

  -> 103 aur 104 ka wait karo!
  -> Buffer karo 105 ko
  -> Jab 103, 104 aa jaaye -> deliver karo order mein: 103, 104, 105

  Solution:
    - Har message pe SEQUENCE NUMBER ya UNIQUE ID lagao
    - Client track kare last processed ID
    - Out-of-order messages ko BUFFER karo
    - Gaps fill hone ka wait karo
    - Duplicates DETECT karo aur DROP karo

  SSE ke liye: Last-Event-ID mechanism help karta hai
  Lekin server ko events STORE karna padta hai replay ke liye

-----------------------------
11d. BACKPRESSURE & FLOW CONTROL
-----------------------------

Problem: Client server ki speed se data process nahi kar pa raha
  - Mobile user slow network pe
  - Browser tab background mein
  - Client heavy processing kar raha

  Server generating: 100 msgs/sec
  Client processing:  10 msgs/sec
  Send buffer: GROWING... GROWING... GROWING... OVERFLOW!

  Options:
    1. DROP low-priority messages (stock ticker -> sirf latest price bhejo)
    2. AGGREGATE/BATCH updates (multiple updates ko ek mein combine)
    3. DISCONNECT slow client (chat app -> client ko hatao)

  Konsa option? -> USE CASE pe depend karta hai!
    - Stock ticker: Drop old prices (latest enough hai)
    - Chat app: Har message deliver karna zaroori (disconnect if too slow)


=============================================================================
SECTION 12: AUTHENTICATION ON LONG-LIVED CONNECTIONS
=============================================================================

Normal HTTP:
  Har request ke saath token/cookie jaata hai
  Server har baar check karta hai -> Simple!

Long-lived connection (SSE/WebSocket):
  Ek baar authenticate hota hai connection time pe
  Connection ghanton tak open reh sakta hai
  Token expire ho sakta hai beech mein!
  "Header toh dubara bhejte nahi" -> Problem!

Solutions:
  1. Handshake pe authenticate karo
  2. Token near expiry -> client refreshed token in-band message mein bhej
     ya server connection close kare -> client naye token se reconnect

SECURITY CONCERNS:

  Concern              | Risk                    | Solution
  ---------------------|-------------------------|------------------------
  Transport encryption | Clear text mein token   | WSS (secure WebSocket)
                       |                         | HTTPS for SSE, NEVER ws://
                       |                         |
  Cross-site hijacking | Malicious page          | Origin header validate
  (CSWSH)              | WebSocket open kare     | karo handshake pe
                       | user ke cookies se      | Prefer token over cookie
                       |                         |
  Token leakage        | SSE: EventSource custom | Short-lived tokens use
                       | header set nahi kar     | karo ya cookie use karo
                       | sakti -> query string   | streaming endpoint ke liye
                       | mein token -> logs mein |                         |
                       | aa sakta hai            |

  CSWSH Important Note:
    Browser same-origin policy WebSocket handshake ko BLOCK nahi karta!
    Aur cookies automatically bhej deta hai!
    Isliye Origin header check KARNA PADTA HAI!
    Sirf cookie-based auth = DANGEROUS


=============================================================================
SECTION 13: LOAD BALANCING & ZERO-DOWNTIME DEPLOYS
=============================================================================

STATELESS HTTP:
  Load balancer kisi bhi node ko request bhej sakta hai -> Easy

STATEFUL WEBSOCKETS:
  Connection ek baar establish ho gaya toh MOVE nahi hoti!
  Ye game change karta hai.

LOAD BALANCER SETUP:
  - WebSocket HTTP se start hota hai with Upgrade header
  - Load balancer ko Layer 7 pe kaam karna chahiye (HTTP aware)
  - ya Layer 4 pe (TCP proxy)

  COMMON BUG:
    "WebSockets local pe kaam karte hain, production mein nahi"
    Reason: Purane proxies Upgrade header strip kar dete hain
    Aur connection silently HTTP pe reh jaati hai!

  IDLE TIMEOUT:
    - Balancer 60 sec baad quiet connection band kar dega
    - Heartbeats isliye bhi zaroori hain (connection alive rakhne ke liye)

ZERO-DOWNTIME DEPLOY:
  Problem:
    Server restart -> uss server ki SAARI connections ek saath tootengi
    -> Saare clients ek saath reconnect karenge
    -> THUNDERING HERD!

  Solutions:

    1. CONNECTION DRAINING:
       - Naye connections accept band karo
       - Existing connections ko grace period do
       - Phir gracefully shutdown karo

    2. STAGGERED ROLLOUTS:
       - Ek baar mein sirf fraction of servers restart
       - Isliye sirf ek portion of connections tootengi

    3. CLIENT-SIDE BACKOFF WITH JITTER:
       - Reconnect karte waqt random delay
       - Sab ek saath nahi aayengi


=============================================================================
SECTION 14: REAL-WORLD EXAMPLES
=============================================================================

  System          | Transport    | Reason
  ----------------|--------------|-------------------------------------------
  Slack           | WebSocket    | High-frequency bidirectional (messages,
                  |              | presence, typing)
                  |              |
  Discord         | WebSocket    | Gateway connection with heartbeats,
                  |              | sequence numbers, resume protocol
                  |              |
  WhatsApp        | Persistent   | Long-lived TCP connection (customized
                  | Socket       | XMPP). WhatsApp Web = WebSocket
                  |              |
  Figma           | WebSocket    | Multiplayer editing = continuous two-way
                  |              | document sync
                  |              |
  Coinbase/       | WebSocket    | Low-latency order book + trade updates
  Binance         |              | Subscription-based control
                  |              |
  ChatGPT /       | SSE          | Token-by-token streaming = one-way
  LLM UIs         |              | server -> client = SSE perfect fit

KEY INSIGHTS:

  Discord ka pattern:
    WebSocket -> live stream ke liye
    Heartbeats -> dead connection detect
    Sequence numbers -> missed events replay
    Transport ALONE delivery guarantee nahi karta!

  LLM streaming ka pattern:
    Sabse naya real-time feature (AI streaming)
    SSE use karta hai, WebSocket NAHI!
    Kyun? Data sirf ek taraf jaata hai (server -> client)
    SSE simple hai aur kaafi hai


=============================================================================
SECTION 15: BEYOND THE 4 APPROACHES
=============================================================================

-----------------------------
15a. PUSH NOTIFICATIONS (APNs & FCM)
-----------------------------

Problem:
  Jab mobile app BACKGROUND ya CLOSED hai:
  - OS app ko suspend kar deta hai
  - Connections tod deta hai battery bachane ke liye
  - SSE/WebSocket/Long Polling SAB kaam band!

Solution:
  APNs (Apple Push Notification service)
  FCM (Firebase Cloud Messaging)

  App Server
      |
      | "Ye notification bhejo"
      v
  [APNs / FCM]  <- Ye maintain karta hai apna connection device se
      |
      | "Wake up! Notification hai!"
      v
  [Device] <- App wake up hota hai

  REAL APP strategy:
    Foreground mein -> SSE / WebSocket (live connection)
    Background mein -> Push notifications (APNs/FCM)

-----------------------------
15b. WebRTC
-----------------------------

Use case: Video calls, voice, screen sharing
  - Peer-to-peer direct connection between browsers
  - Typically UDP based (speed over guaranteed delivery)
  - Lowest possible latency
  - Complex setup (signaling server, STUN/TURN)

  Client A <---> [Direct P2P Connection] <---> Client B
                 (UDP, fallback to relay)

  Ye 4 approaches ko REPLACE nahi karta, COMPLEMENT karta hai
  Real-time audio/video ke liye WebRTC
  General data updates ke liye SSE/WS


=============================================================================
SECTION 16: KEY TAKEAWAYS (Interview ke liye!)
=============================================================================

4 Approaches Progression:

  Short Polling  -->  Long Polling  -->  SSE  -->  WebSocket
  (Simplest)                                      (Most Complex)
  (Client poll)                                   (Bidirectional)
  (Stateless)                                     (Stateful)

IMPORTANT PRINCIPLES:

  1. REQUIREMENTS PEHLE:
     Latency, directionality, message rate, fanout,
     ordering, replay -> ye sab define karo PEHLE
     phir transport choose karo

  2. SIMPLE SE START KARO:
     Agar short polling kaam karta hai -> use karo!
     Sabse simple jo kaam kare = best solution

  3. SSE UNDERRATED HAI:
     Data server -> client flow kare? SSE use karo
     WebSockets se simple hai + browser reconnect support

  4. WEBSOCKETS POWERFUL BUT COMPLEX:
     Sirf tab use karo jab bidirectional + high-frequency chahiye
     Scaling aur operational complexity SIGNIFICANT hai

  5. TRANSPORT != DELIVERY SEMANTICS:
     Reliable delivery, ordering, replay ke liye ADD karo:
     - Sequence IDs
     - Durable logs
     - Cursors
     - Deduplication
     - Backpressure

  6. FAILURE KE LIYE PLAN KARO:
     Connections tootengi -> implement karo:
     - Exponential backoff with jitter
     - Heartbeats for health detection
     - Message ordering for reliability


=============================================================================
SECTION 17: INTERVIEW CHEAT SHEET (Quick Reference)
=============================================================================

  Puchho khud se:
    Q: "Latency target kya hai?"
    A: Milliseconds? Seconds? Eventually?

    Q: "Data kis direction mein flow karta hai?"
    A: Server->Client? Client->Server? Dono?

    Q: "Kitne clients? Kitne messages/sec?"
    A: Scale decide karta hai cost

    Q: "Messages durable hone chahiye?"
    A: Pub/Sub ephemeral hai, replay ke liye log chahiye

    Q: "Ordering chahiye?"
    A: Chat, edits, trades -> sequence IDs

    Q: "Reconnect pe kya hoga?"
    A: Resume tokens, deduplication, backoff

    Q: "Kya updates drop ho sakte hain?"
    A: Presence/prices -> haan. Chat messages -> nahi

  Common Mistake:
    "WebSockets sabse powerful hain, chalo WebSockets use karte hain"
    -> GALAT! Pehle requirements samjho, phir transport choose karo

  Golden Rule:
    "The best approach is the one that solves your problem
     with the least complexity."
    -> Simple se complex ki taraf jao, tab jab zaroorat ho

=============================================================================
                        END OF NOTES
=============================================================================

Bhai, ye notes padh le toh Real-time Updates ka interview
topic clear ho jaayega. Bas ye yaad rakh:

  1. Requirements define karo (latency, direction, scale)
  2. Simplest approach jo kaam kare wahi choose karo
  3. SSE underrated hai - server->client ke liye best
  4. WebSockets sirf jab truly bidirectional chahiye
  5. Transport ke upar reliability layer add karna padta hai
  6. Failure planning mandatory hai

All the best for your interview! 🚀
=============================================================================
*/
